## GPT prompting analysis


In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import csv

In [42]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

## Vastustega df

In [3]:
df1 = pd.read_csv("../gpt_output/n80_examples_large_v1_gpt_v2_10K_b10_v1.csv", encoding="utf-8", sep=",")

In [4]:
df1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
0,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
1,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'.",yes,NaN
2,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,NaN
3,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN,yes,NaN
4,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'.",yes,NaN
9996,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'.",yes,NaN
9997,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'.",yes,NaN
9998,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN,yes,NaN


### yes % katse 1

In [6]:
yes = len(df1[df1["classification"]=="yes"])
#no = len(df1[df1["classification"]=="no"])
#answered = yes+no
yes_percent = yes*100/len(df1)
yes_percent

64.74

### yes % katse 2

In [5]:
yes = len(df1[df1["classification2"]=="yes"])
#no = len(df1[df1["classification2"]=="no"])
#answered = yes+no
yes_percent = yes*100/len(df1)
yes_percent

86.94

## yes -> no %

In [13]:
yes_2_no = len(df1[(df1["classification"]=="yes") & (df1["classification2"]=="no")])
print(f"arvuliselt yes->no : {yes_2_no}")
print(f"protsendiliselt yes->no : {yes_2_no*100/len(df1)}%")

arvuliselt yes->no : 8
protsendiliselt yes->no : 0.08%


## no -> yes %

In [15]:
no_2_yes = len(df1[(df1["classification"]=="no") & (df1["classification2"]=="yes")])
print(f"arvuliselt no->yes : {no_2_yes}")
print(f"protsendiliselt no->yes : {no_2_yes*100/len(df1)}%")

arvuliselt no->yes : 2056
protsendiliselt no->yes : 20.56%


## yes -> no näited

In [17]:
yes2no_ex = df1[(df1["classification"]=="yes") & (df1["classification2"]=="no")]
# explanation=esimese korra yes põhjendus, explanation2=teise korra no põhjendus
yes2no_ex = yes2no_ex[["form", "verb", "verb_compound", "morph_case", "sentence", "explanation", "explanation2"]]
yes2no_ex

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
841,suunda,avanema,NaN,adit,"Näiteks Kassitoomel avaneb ilusaid vaateid igasse suunda , kuid ühtki pinki istumiseks seal ei ole .",NaN,"The phrase 'suunda' refers to a direction, which is not a specific physical location or spatial context, so it is not considered an adverbial of place."
1794,Hirve,tungima,NaN,adit,"05.03.2003 tungis Tallinnas Hirve 27 juures Andresele ( s. 1979 ) kallale Eduard ( s. 1985 ) , kes peksis teda , tekitades Eduardile raskeid tervisekahjustusi .",Hirve was classified as 'yes' because it refers to a specific place in Tallinn.,The phrase 'Hirve' was classified as 'no' because it refers to a proper noun (a name) without specifying a tangible location.
2224,traktoriroolis,lõhkema,NaN,in,"Kellele jätta elutöö , mille eest on makstud parimal juhul silmanägemise ja kõ-veraks kulunud kehaga , halvimal juhul aga kevadkülvikampaania ajal traktoriroolis lõhkenud südamega ?","The phrase 'traktoriroolis' refers to being at the wheel of a tractor, which suggests a specific position or location, so it was classified as location.",The phrase 'traktoriroolis' (at the tractor's wheel) is not classified as an adverbial of place because it refers to a specific role or situation rather than a physical location.
3123,vanni,mahtuma,ära,adit,"Kaalujälgijatega olid tänavuse võistluse kandidaadid ühinenud erinevail põhjustel : kes ei mahtunud enam vanni või pükstesse ära , kes tahtis vastassoole muljet avaldada , kel muutus liikumine päev päeva järel vaevalisemaks , kes tuli sõbra kutsel .",NaN,"The phrase 'vanni' was classified as 'no' because it does not describe a location or place where an action occurs, but is part of a metaphor or expression about fitting into something."
3271,Kairis,möllama,NaN,in,"Kairis möllasid korterisse sisenedes vastakad tunded : « Vaatasin , autot maja ees pole .",NaN,The phrase 'Kairis' was classified as 'no' because it functions as the subject of the sentence and does not indicate a place.
3858,Mooni,kihutama,NaN,adit,"Noorsoopolitseinik Rene Luide andmetel jäi vahele ka kaks joobes roolikeerajat , kellest üks kihutas Mooni 100 asuva kooli juurde paigaldatud teekünniste juures .",NaN,"The phrase 'Mooni' is not adverbial of place because it refers to a proper name or entity, rather than indicating a specific location."
6389,seljas,sõitma,ringi,in,Nii palju loodusesõpru pole jalgrataste seljas üheskoos ringi sõitnud 1992. aastast alates .,"The phrase 'seljas' refers to the position (on bicycles) that people are riding in and not a physical location, so it is classified as 'yes'.","The phrase 'seljas' refers to being on something (e.g., 'on bicycles') and does not indicate a physical location, thus it is not classified as an adverbial of place."
7370,Tartu-,saatma,NaN,adit,"Rappima on veel liiga pehme sõna väljendamaks seda kahju , mida esmaspäevane äikesetorm koos rahega Tartu- ja Jõgevamaal korda saatis .",NaN,"The phrase 'Tartu-' is part of a compound word and does not specify a location within the context of the sentence; hence, it is not adverbial of place."


### probleemsed:

1. metafoorilised: vanni ära mahtuma
2. nimed (isikud/koahnimed): Hirve, Kairis, Mooni, Tartu- 
3. objektid: jalgratta seljas
4. suunad

## no -> yes näiteid

In [40]:
no2yes_ex = df1[(df1["classification"]=="no") & (df1["classification2"]=="yes")]
# explanation=esimese korra yes põhjendus, explanation2=teise korra no põhjendus
no2yes_ex = no2yes_ex[["form", "verb", "verb_compound", "morph_case", "sentence", "explanation", "explanation2"]]
len(no2yes_ex)

2056

In [ ]:
no2yes_ex

### no->yes gpt põhjendused:

1. suund
2. lähtekoht/source
3. sihtkoht
4. event
5. "specific/particular place"
6. spatial
7. programm (telekanal, saade, programm)

In [31]:
no2yes_ex[~no2yes_ex["explanation2"].isna()]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
120,EMU-sse,andma,NaN,ill,Samas annab ka Euroopa Rahandusinstituut Frankfurdis oma hinnangu EMU-sse pürgijatele .,The phrase 'EMU-sse' was classified as 'no' because it refers to a monetary union (European Monetary Union) rather than a physical location.,Classified as adverbial of place ('yes') because 'EMU-sse' specifies a destination related to the action of striving or aspiring.
121,poliitikutesse,kaduma,NaN,ill,Kõigepealt kadus Soomes usk poliitikutesse .,"‘poliitikutesse’ was classified as not a location because it refers to a group of people (politicians), not a geographic place.",Classified as adverbial of place ('yes') because 'poliitikutesse' indicates the direction or focus of the loss of faith.
130,riigieelarvesse,tooma,tagasi,ill,"Kas komisjonis tuli ka jutuks selline temaatika , et kui mina saan tasulise arstiabi kasutamisel maksusoodustust , siis arsti sissetulek suureneb ja tema maksab minu raha pealt jällegi tulumaksu , mis toob tulumaksu jälle uuesti riigieelarvesse tagasi , aga selle tulemusel ei ole minu n-ö tulumaksuga maksustatav raha kulutatud mitte hilpude peale , vaid õilsal eesmärgil , tervise parandamiseks ?","The phrase 'riigieelarvesse' was classified as 'no' because it refers to the state budget, not a location.","The word 'riigieelarvesse' indicates direction towards the state budget, which is a location in an abstract sense, and therefore, it is classified as an adverbial of place."
141,ARK-s,omama,NaN,in,""" Käis tihe koostöö , Kaljusaar sai autokooli sekretärina tunnistusi võltsida , Stolts omas kontakti Pärnu ARK-s , igaühel oli oma roll ja tööjaotus , "" ütles prokurör Hirvoja .",ARK-s refers to a specific institution and not a geographic or physical location.,"The phrase 'ARK-s' specifies a particular place (Pärnu ARK), making it an adverbial of place."
170,mängumaale,pöörduma,tagasi,all,"Nii polegi ime , kui need vabastatud pettuvad ja pöörduvad tagasi vanale mängumaale .",The phrase 'mängumaale' refers to a figurative playground or realm rather than an actual geographical location.,"The phrase 'mängumaale' refers to a specific place or location where events occur, so it is classified as an adverbial of place."
...,...,...,...,...,...,...,...
9701,televisioonis,seiklema,NaN,in,"Imemees MacGyver on seigelnud aastate eest Soome televisioonis , kus MTV 3 praegu vanu seeriaid üle kordab .","'televisioonis' is not classified as a location because it refers to a medium (television), not a geographic entity.","The word 'televisioonis' refers to a location (television), so it is classified as an adverbial of place."
9760,Pajuneni,saama,NaN,adit,Läbi Pajuneni sai Ingman ka Spice Ice jäätisekohvikuid haldava OÜ Jäätisefriik omanikuks .,"The phrase 'Pajuneni' refers to a person or entity in this context, not a geographical location.",The phrase 'Pajuneni' is classified as an adverbial of place because it refers to a location associated with how Ingman became the owner.
9810,mängus,tervitama,NaN,in,"Igas mängus tervitavad tribüünid teda , skandeerides "" Pooooooooooooooom "" .",The phrase 'mängus' refers to an event or activity (a game) and not a specific physical location.,"The phrase 'mängus' refers to the location or context of an event (in the game), so it is classified as an adverbial of place."
9841,raamatupidamisse,kandma,NaN,adit,Liikmesriikide sekkumisametid kannavad sellistest toimingutest tulenevad kulutused igal kuul raamatupidamisse Nõukogu Liidule antava hädaabi kulutustena .,"The phrase 'raamatupidamisse' refers to an accounting process or department, not a location.","The phrase 'raamatupidamisse' indicates a place where the costs are recorded, classifying it as an adverbial of place."


In [47]:
direction = no2yes_ex[no2yes_ex["explanation2"].fillna("").str.contains("direction")]
direction

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
121,poliitikutesse,kaduma,NaN,ill,Kõigepealt kadus Soomes usk poliitikutesse .,"‘poliitikutesse’ was classified as not a location because it refers to a group of people (politicians), not a geographic place.",Classified as adverbial of place ('yes') because 'poliitikutesse' indicates the direction or focus of the loss of faith.
130,riigieelarvesse,tooma,tagasi,ill,"Kas komisjonis tuli ka jutuks selline temaatika , et kui mina saan tasulise arstiabi kasutamisel maksusoodustust , siis arsti sissetulek suureneb ja tema maksab minu raha pealt jällegi tulumaksu , mis toob tulumaksu jälle uuesti riigieelarvesse tagasi , aga selle tulemusel ei ole minu n-ö tulumaksuga maksustatav raha kulutatud mitte hilpude peale , vaid õilsal eesmärgil , tervise parandamiseks ?","The phrase 'riigieelarvesse' was classified as 'no' because it refers to the state budget, not a location.","The word 'riigieelarvesse' indicates direction towards the state budget, which is a location in an abstract sense, and therefore, it is classified as an adverbial of place."
191,haigustesse,andma,NaN,ill,"Samas annavad arstid nõu lapse esmase haigestumise korral , eriti allergoloogilistesse , kopsu- ja kardioloogilistesse haigustesse .","The phrase 'haigustesse' refers to diseases or illnesses, not a geographic or physical location, thus it was classified as 'no'.","The phrase 'haigustesse' is classified as adverbial of place because it implies direction or movement towards an abstract location, specifically illnesses."
280,teosesse,mahtuma,ära,ill,"Teiseks , et papi elu ei mahu ühte teosesse ära .","The phrase 'teosesse' refers to a piece of work or creation, not a physical or geographical location, so it is not classified as location.","The phrase 'teosesse' was classified as adverbial of place ('yes') because it indicates a target or direction, referring to a specific location (in this case, into the work)."
491,koondisesse,tahtma,NaN,ill,Levadia tahaks koondisesse,"The phrase 'koondisesse' refers to being part of a group, not to a physical location, hence classified as 'no'.",It is classified as adverbial of place because 'koondisesse' (into the team) specifies a direction or destination associated with a place.
500,sinnasamusesse,keerama,NaN,ill,"Oleks olnud dekreet viinast , dekreet pesupulbrist või üleüldse konkreetne jutt : kuradi Gorba , keerab siin riigi sinnasamusesse , aitab pullist , nüüd on jälle jagamatu .","The phrase 'sinnasamusesse' is a vague or colloquial reference to a state or condition, not a specific geographical location, so it is classified as 'no'.","The phrase 'sinnasamusesse' was classified as adverbial of place ('yes') because it denotes a specific, albeit figurative, location or direction."
590,lainesse,laskma,NaN,ill,( Laevaremondimeeste prits lasi autopleki lainesse : ),The word 'lainesse' does not refer to a geographical location; it describes a state or form of the car’s surface.,"The phrase 'lainesse' was classified as 'yes' because it describes a specific direction or position, indicating a place-like movement or destination."
770,toodetesse,ütlema,NaN,ill,"Mustajoki ütleb end umbuslikult suhtuvat ka kõikidesse püruvaati sisaldavatesse toodetesse , sest uuringud püruvaadi ainevahetust kiirendavate ja rasva põletava toime vallas pole veel kaugeltki täielikud .","The word 'toodetesse' refers to products, which are objects and not locations.","The phrase 'toodetesse' indicates a direction or location, making it adverbial of place."
1440,tippudesse,kihutama,NaN,ill,"Laiale publikule tutvustas Sonique'i tema läbilöögihitt "" Feels So Good "" , mis tütarlapse hoobilt ka edetabelite tippudesse kihutas .","The term 'tippudesse' refers to achieving a peak or top position metaphorically, not a physical location, hence classified as 'no'.","The phrase 'tippudesse' indicates a direction or location (to the tops) where the subject's action culminates, fitting the definition

In [37]:
other1 = no2yes_ex[(~no2yes_ex["explanation2"].fillna("").str.contains("direction")) & (~no2yes_ex["explanation2"].isna())]
other1

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
120,EMU-sse,andma,NaN,ill,Samas annab ka Euroopa Rahandusinstituut Frankfurdis oma hinnangu EMU-sse pürgijatele .,The phrase 'EMU-sse' was classified as 'no' because it refers to a monetary union (European Monetary Union) rather than a physical location.,Classified as adverbial of place ('yes') because 'EMU-sse' specifies a destination related to the action of striving or aspiring.
141,ARK-s,omama,NaN,in,""" Käis tihe koostöö , Kaljusaar sai autokooli sekretärina tunnistusi võltsida , Stolts omas kontakti Pärnu ARK-s , igaühel oli oma roll ja tööjaotus , "" ütles prokurör Hirvoja .",ARK-s refers to a specific institution and not a geographic or physical location.,"The phrase 'ARK-s' specifies a particular place (Pärnu ARK), making it an adverbial of place."
170,mängumaale,pöörduma,tagasi,all,"Nii polegi ime , kui need vabastatud pettuvad ja pöörduvad tagasi vanale mängumaale .",The phrase 'mängumaale' refers to a figurative playground or realm rather than an actual geographical location.,"The phrase 'mängumaale' refers to a specific place or location where events occur, so it is classified as an adverbial of place."
200,Rahvusvahelisele,naasma,NaN,all,USA kosmosesüstik Atlantis naasis retkelt Rahvusvahelisele Kosmosejaamale ning maandus kell 09.57 Eesti aja järgi Maal .,The phrase 'Rahvusvahelisele' was classified as not a location because it is an adjective describing 'Kosmosejaamale' (International Space Station) and does not itself specify a place.,'Rahvusvahelisele' was classified as 'yes' because it specifies a location related to the International Space Station.
211,eraettevõtlusesse,laskma,NaN,ill,Üldiselt vist riigid luurejuhi kohalt helgemaid päid eraettevõtlusesse ei lase ?,"The term 'eraettevõtlusesse' refers to private entrepreneurship, which is a sector, not a geographical location.","The phrase 'eraettevõtlusesse' denotes a destination or location for the action (into the private sector), qualifying it as an adverbial of place."
...,...,...,...,...,...,...,...
9690,juhikabiinist,avastama,NaN,el,Tollijärelevalve osakonna töötajad kontrollisid Peterburi liinil sõitva Fantaasia meeskonna ja laevavarude liikumist reisisadama D-terminalis ning avastasid laevalt pardavarusid täiendamast naasnud veoki juhikabiinist kahte kilekotti pakendatuna Venemaa maksumärke kandvad sigaretid ja närimistubaka karbid .,"The phrase 'juhikabiinist' was classified as 'no' because it refers to the driver's cabin of a vehicle, not a general geographical place.","The phrase 'juhikabiinist' was classified as 'yes' because it specifies the place (driver's cabin) from which the items were found or originated, making it adverbial of place."
9701,televisioonis,seiklema,NaN,in,"Imemees MacGyver on seigelnud aastate eest Soome televisioonis , kus MTV 3 praegu vanu seeriaid üle kordab .","'televisioonis' is not classified as a location because it refers to a medium (television), not a geographic entity.","The word 'televisioonis' refers to a location (television), so it is classified as an adverbial of place."
9760,Pajuneni,saama,NaN,adit,Läbi Pajuneni sai Ingman ka Spice Ice jäätisekohvikuid haldava OÜ Jäätisefriik omanikuks .,"The phrase 'Pajuneni' refers to a person or entity in this context, not a geographical location.",The phrase 'Pajuneni' is classified as an adverbial of place because it refers to a location associated with how Ingman became the owner.
9810,mängus,tervitama,NaN,in,"Igas mängus tervitavad tribüünid teda , skandeerides "" Pooooooooooooooom "" .",The phrase 'mängus' refers to an event or activity (a game) and not a specific physical location.,"The phrase 'mängus' refers to the location or context of an event (in the game), so it is classified as an adverbial of place."


In [58]:
origin = other1[(other1["explanation2"].str.contains("origin")) | (other1["explanation2"].str.contains("source"))]
origin

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
671,Helitaustast,kostma,NaN,el,"Helitaustast kostab unenäolisi lausekatkeid : Tasa-tasa-tasa ! , keegi tüdruk hüüab midagi - siis hääbub tüdruku hääl meditatiivsesse helijadasse .","The phrase 'Helitaustast' refers to a sound background, which does not denote a physical place, so it was classified as 'no'.","The phrase 'Helitaustast' specifies the origin or source of a sound, making it an adverbial of place."
1070,KGB-st,tulema,ära,el,Vladimir tuli ära KGB-st ja tõusis linnavalitsuses 1994. aastaks linnapea asetäitjaks .,The word 'KGB-st' refers to an organization rather than a physical location.,"The phrase 'KGB-st' specifies the origin or source of Vladimir and implies a place, thus it is classified as an adverbial of place ('yes')."
1190,NASA-st,kostma,NaN,el,"Mida enam kostab NASA-st hääli , et kosmoseteleskoop Hubble lõpetab tegevuse , seda enam saabub Hubble'ist uudiseid .","The word 'NASA-st' refers to an organization rather than a specific geographic location, so it is classified as 'no'.",The phrase 'NASA-st' indicates the origin or source of sounds and is therefore classified as an adverbial of place.
1561,Lainest,tulema,ära,el,"Aga kui ma Lainest ära tulin , liitus sellega Tõnis Mägi .","The phrase 'Lainest' does not refer to a specific location but possibly a name, so it was classified as not a location ('no').","The phrase 'Lainest' specifies the location or origin the subject is leaving, answering the question 'where from,' so it is classified as an adverbial of place."
2371,EGeenist,lahkuma,NaN,el,"Millised olid need vastuolud , miks lahkusid EGeenist Andres Metspalu ja Jaanus Pikani ?","The term 'EGeenist' references an organization or company, not a specific location, so it is not classified as a location.","The phrase 'EGeenist' specifies a location or origin related to the action of 'leaving', which classifies it as an adverbial of place."
2520,Viimasest,importima,NaN,el,Viimasest impordin näiteks kristalli .,"The word 'Viimasest' is a reference to the previous entity and does not specify a location, so it is classified as 'no'.","It was classified as 'yes' because 'Viimasest' indicates the origin or source (a place or situation) from which something is imported, thus serving as an adverbial of place."
4491,sealihast,importima,NaN,el,"Sontsi sõnul annab importliha hinnatõus Lätis võimaluse tõsta ka kodumaise liha hinda , sest Läti impordib suure osa tarbitavast sealihast .",The word 'sealihast' refers to pork meat and not a geographical location.,"The word 'sealihast' indicates the origin of the pork and implies a location-based idea, hence it is considered an adverbial of place."
4770,kolmainsusest,olema,ära,el,"Kui sellest kolmainsusest on midagi korrast ära , tekivadki hambaaugud .",The phrase 'kolmainsusest' refers to a spiritual concept and not a location.,"The phrase 'kolmainsusest' refers to the source or origin of something, thus functioning as an adverbial of place."
4830,koalitsioonileppest,võtma,välja,el,"Tuleva aasta teisel poolel peaksid kallinema toiduained , sest koond- ja maamehed võtsid koalitsioonileppest välja kaitsetolle keelustava punkti .","The phrase 'koalitsioonileppest' pertains to a coalition agreement, which is not a location.","The phrase 'koalitsioonileppest' indicates the origin or source of something ('from the coalition agreement'), and origin within a context can denote a place, thus classified as adverbial of place."
5861,meditsiinist,voolama,välja,el,Nii voolab riiklikust meditsiinist elujõud muudkui välja .,"The word 'meditsiinist' refers to the medical field, which is an abstract concept and not a location.","The phrase 'meditsiinist' indicates the origin or source location ('out of the medical system'), which qualifies it as an adverbial of place."


In [44]:
destination = other1[other1["explanation2"].str.contains("destina")]
destination

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
120,EMU-sse,andma,NaN,ill,Samas annab ka Euroopa Rahandusinstituut Frankfurdis oma hinnangu EMU-sse pürgijatele .,The phrase 'EMU-sse' was classified as 'no' because it refers to a monetary union (European Monetary Union) rather than a physical location.,Classified as adverbial of place ('yes') because 'EMU-sse' specifies a destination related to the action of striving or aspiring.
211,eraettevõtlusesse,laskma,NaN,ill,Üldiselt vist riigid luurejuhi kohalt helgemaid päid eraettevõtlusesse ei lase ?,"The term 'eraettevõtlusesse' refers to private entrepreneurship, which is a sector, not a geographical location.","The phrase 'eraettevõtlusesse' denotes a destination or location for the action (into the private sector), qualifying it as an adverbial of place."
220,öösse,ootama,NaN,ill,"Ootame sind kuuma öösse - reedel , 12. oktoobril uues CLUB TALLINNAS !!!",The phrase 'öösse' refers to a concept (night) rather than a specific location.,"The phrase 'öösse' indicates a destination or location, hence it is classified as an adverbial of place."
521,ÜRO-sse,lõpetama,NaN,ill,"1994. aastal lõpetas pärast Palau iseseisvumist ja vastuvõtmist ÜRO-sse oma missiooni Hooldusnõukogu ( Trusteeship Council ) , mille ülesanne oli tagada koloniaalvalduste üleminek suveräänsusele .","The phrase 'ÜRO-sse' refers to the United Nations (ÜRO in Estonian), which is an organization and not a specific geographical location, hence classified as 'no'.","The phrase 'ÜRO-sse' specifies a location, referring to the United Nations as the destination of admission, so it is classified as adverbial of place."
1430,moemeediasse,leidma,NaN,ill,90ndate lõpuaastail leidsid äraspidised kodukaadrid uuesti tee moemeediasse .,"The phrase 'moemeediasse' refers to fashion media, which is an industry and not a geographic location, so it was classified as 'no'.","The phrase 'moemeediasse' was classified as 'yes' because it indicates the destination or place where the 'kodukaadrid' were found, making it an adverbial of place."
2150,erakonnakassasse,maksma,NaN,ill,Palju sotsid europarlamendist erakonnakassasse raha maksavad ?,"The phrase 'erakonnakassasse' refers to a party's financial account and not a geographic or physical location, so it was classified as not a location.","The phrase 'erakonnakassasse' was classified as 'yes' because it indicates a physical destination, referring to the party's cashbox."
2241,Ringhäälingunõukogusse,hakkama,NaN,ill,Ringhäälingunõukogusse hakkab kuuluma nii koalitsiooni kui opositsiooni esindajaid .,"The word 'Ringhäälingunõukogusse' refers to a broadcasting council, which is an organization, not a location.","The phrase 'Ringhäälingunõukogusse' indicates a location or destination where people will belong, making it adverbial of place."
2671,seadustessegi,leidma,NaN,ill,"Ometi käib tema juures iga nädal koos ministritest koosnev Riiginõukogu , kus Margrethe asjatundlikud märkused on sageli leidnud tee hilisematesse seadustessegi .","The phrase 'seadustessegi' refers to laws and not a geographic or physical location, so it is classified as not a location.","The phrase 'seadustessegi' indicates the destination or location where Margrethe's remarks have been incorporated, thus making it an adverbial of place."
3210,Magicusse,tahtma,NaN,ill,"Aga Stevie ütles , et ta ei taha Magicusse !","The phrase 'Magicusse' appears to be a name or an abstract reference, not a physical location, hence classified as 'no'.","The phrase 'Magicusse' indicates a destination or location, making it an adverbial of place."
3600,võistlustulle,pöörduma,tagasi,adit,Mõnede lehtede arvates pöördub Schumacher tagasi võistlustulle juba Malaisia GP-l nädala pärast .,"võistlustulle refers to participation in a competition, not a physical location.",The phrase 'võistlustulle' is classified as an adverbial of place because it indicates the destination or location associated with Schumacher's return.


In [45]:
event = other1[other1["explanation2"].str.contains("event")]
event

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
170,mängumaale,pöörduma,tagasi,all,"Nii polegi ime , kui need vabastatud pettuvad ja pöörduvad tagasi vanale mängumaale .",The phrase 'mängumaale' refers to a figurative playground or realm rather than an actual geographical location.,"The phrase 'mängumaale' refers to a specific place or location where events occur, so it is classified as an adverbial of place."
690,joonisfilmides,seiklema,NaN,in,"Heiki Ernitsa viimaste aastate tööna valminud joonisfilmides seiklevad koerad , jänesed , lepatriinud ja muud mutukad .",The phrase 'joonisfilmides' refers to animated films and is not a specific location.,"The phrase 'joonisfilmides' was classified as 'yes' because it specifies the location where the actions or events occur, i.e., within the animated films."
2270,eestlastele,valguma,NaN,all,Ei mõtle siin mitte niivõrd 1940. aastal eestlastele idast kaela valgunud tapmiste ja röövimiste laviini .,"The phrase 'eestlastele' refers to people from Estonia and not to a specific geographic or physical location, so it was classified as 'no'.","The phrase 'eestlastele' specifies the target group (to whom the events are directed) and indirectly points to a nation or people, making it adverbial of place."
2600,mängus,varisema,kokku,in,"20aastane ja 202 cm pikkune Eesti koondise kandidaat varises kokku möödunud laupäeval mängus Grayson College i vastu , kui McLennan juhtis 28 : 25.","The word 'mängus' refers to an event (a game), not a specific physical location.","The phrase 'mängus' specifies the location of the event described (the game against Grayson College), which makes it an adverbial of place."
2890,hilistalvelumel,vedelema,NaN,ad,"Ning küllap on nad õnnega koos , kui hilistalvelumel vedeleb mõni priske metskitsekorjus .",The word 'hilistalvelumel' describes late-winter snow but does not indicate a specific location.,"The phrase 'hilistalvelumel' indicates the location where the event of 'vedelevad' (lying around) takes place, making it adverbial of place."
3540,veerandfinaalis,lõhkema,NaN,in,"Meeste korvpalli EM-võistlustel Pariisis lõhkes esimene üllatuspomm neljapäeva viimases veerandfinaalis ja teine eile esimeses poolfinaalis : Hispaania võitis kullapretendendiks peetud Leedut 74 : 72 ja kurvastas Pariisi publikut , lüües Prantsusmaad 70 : 63.",The phrase 'veerandfinaalis' refers to a stage in a competition and not a geographical or physical location.,The phrase 'veerandfinaalis' was classified as adverbial of place because it indicates the specific location where the event (the upset) occurred.
5540,blokki,jääma,kinni,adit,"Ka viimane punkt tuli siis , kui Keel jäi Araku blokki kinni .","The phrase 'blokki' pertains to a block in the context of a sports game, not a geographic location.",The phrase 'blokki' was classified as adverbial of place ('yes') because it indicates the specific location where the event (Keel staying stuck) occurred.
6320,B-finaalis,ootama,NaN,in,"B-finaalis aga ootavad prantslased ja taanlased , kes on kogu suve MK-etappide finaalis sõudnud - meie pole iial nii kaugele jõudnud . """,B-finaalis is not a specific physical location but rather a competition stage or category.,"The phrase 'B-finaalis' indicates a specific location where the event is happening (the B-final), making it an adverbial of place."
6570,finaalseerias,arenema,NaN,in,Meeste Korvpalliliiga finaalseerias arenesid tasavägiselt vaid kaks esimest kohtumist .,"The word 'finaalseerias' refers to a series of final games, which is an event rather than a location, so it is classified as 'no'.","The phrase 'finaalseerias' specifies the location where the events (the matches) took place, making it an adverbial of place."
6730,komöödias,pakutama,NaN,in,"Kuigi selles komöödias ei pakuta suuremat sorti üllatusi , on see läbilõhki sümpaatne ja lõbus film , milles ei puudu ka sügavamad ning hingekriipivad noodid .","The phrase 'komöödias' refers to a genre or type of film, not a specif

In [51]:
specific_place = other1[(other1["explanation2"].str.contains("specific place")) | other1["explanation2"].str.contains("particular place")]
specific_place

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
141,ARK-s,omama,NaN,in,""" Käis tihe koostöö , Kaljusaar sai autokooli sekretärina tunnistusi võltsida , Stolts omas kontakti Pärnu ARK-s , igaühel oli oma roll ja tööjaotus , "" ütles prokurör Hirvoja .",ARK-s refers to a specific institution and not a geographic or physical location.,"The phrase 'ARK-s' specifies a particular place (Pärnu ARK), making it an adverbial of place."
170,mängumaale,pöörduma,tagasi,all,"Nii polegi ime , kui need vabastatud pettuvad ja pöörduvad tagasi vanale mängumaale .",The phrase 'mängumaale' refers to a figurative playground or realm rather than an actual geographical location.,"The phrase 'mängumaale' refers to a specific place or location where events occur, so it is classified as an adverbial of place."
1720,ettevõttes,töötama,kokku,in,"Kindlustustoetust makstakse registrisse kantud füüsilisest isikust ettevõtjale või äriühingule , kelle ettevõttes töötab kokku kuni 80 inimest ning kelle aasta netokäive ei ületa 100 miljonit krooni .","The word ""ettevõttes"" refers to a company or business, which is not a geographical location.",The phrase 'ettevõttes' was classified as 'yes' because it refers to a specific place (within a company) where people are working.
3790,Internetilogiraamatus,seisma,NaN,in,"Meie tagasihoidlikus varustuses meestel hakkas päris kõhe , kuid pikapeale hakkas pall ühe rohkem ka vastaste väljakupoolel liikuma , ” seisab Lennuki Internetilogiraamatus .",The phrase 'Internetilogiraamatus' was classified as 'no' because it refers to an online logbook or digital record and not a geographical or physical location.,"The phrase 'Internetilogiraamatus' was classified as 'yes' because it indicates the specific place where the action of 'seisab' (is stated) occurs, making it an adverbial of place."
5490,pühamasse,ronima,NaN,ill,"Riigiduumas levitatud avalduses on öeldud , et “ inimene , kes teostas venevastase pöörde televisiooni esimesel kanalil , ronib pühamast pühamasse - lahendama Vene riigi julgeoleku küsimusi ” .","The word 'pühamasse' refers to a sacred place rather than a geographical location, so it was classified as not location ('no').","The phrase 'pühamasse' indicates movement towards a specific place, qualifying it as an adverbial of place."
5711,Blatnoimaailmas,treenima,NaN,in,"Jouni Hiltuneni "" Blatnoimaailmas "" treenivad vene eluaegsed vangid , kes on eneselegi ootamatult surmanuhtlusest pääsenud , keha ja vaimu .","The phrase 'Blatnoimaailmas' refers to a fictional or conceptual place, not a specific geographical location, hence it was classified as 'no'.",The phrase 'Blatnoimaailmas' was classified as 'yes' because it specifies a particular place or world where the action occurs.
5730,mõjusfääri,liikuma,NaN,adit,Politseisiseste motivatsiooni- ja distsipliiniprobleemide tõttu kaotab riik kontrolli politsei üle ja see liigub organiseeritud kuritegelike struktuuride mõjusfääri .,"The phrase 'mõjusfääri' refers to a sphere of influence rather than an actual geographic or physical location, so it is classified as 'no'.","The phrase 'mõjusfääri' indicates a specific place or location where something is being influenced or moved towards, making it an adverbial of place."
5941,kaadris,ilutsema,NaN,in,"Aga kui asi ise on üks igavene udu ja mõistusega tabamatu jutuvada , siis ei päästa asja ka ilus pilt ja kaadris ilutsevad küünlad .","The term 'kaadris' refers to a frame or scene, which is not considered a geographic or physical location.","The phrase 'kaadris' indicates a specific place where the candles are in the frame, so it was classified as an adverbial of place ('yes')."
6130,Postimehesse,keerama,NaN,ill,Olga Tartu keerab taimed vanasse “ Postimehesse ” ning ulatab presidendile .,The phrase 'Postimehesse' refers to a publication or newspaper and not a physical location.,"The phrase 'Postimehesse' indicates a specific place or destination (into 'Postimees'), so it was classified as 'yes'."
6631,

In [54]:
spatial = other1[other1["explanation2"].str.contains("spatial")]
spatial

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
401,dimensioonist,avastama,NaN,el,""" Kui aga hakata midagi pikka aega kordama , siis avastad end justkui teisest dimensioonist .","The phrase 'dimensioonist' refers to another dimension, which is abstract and not a physical or geographical location.","The phrase 'dimensioonist' indicates a spatial concept, referring to a different dimension, making it an adverbial of place."
3100,tsentris,laiutama,NaN,in,"Staatilise lavaruumi tsentris laiutab võimu atribuutika : juutide püha altar , mille kohale saatuslikul hetkel kerkib Nebukadnetsari hiigeltroon .","The word 'tsentris' refers to a central point or position in a spatial context but does not denote a specific geographical location, so it is not classified as a location.","The phrase 'tsentris' indicates the central location within the static stage space, which relates to the spatial setting, making it an adverbial of place."
3730,põlveõndlasse,looma,NaN,ill,Üks meestest lõi talle selja tagant jalaga põlveõndlasse .,"The term 'põlveõndlasse' refers to a body part and not a location, so it was classified as 'no'.","The phrase 'põlveõndlasse' specifies a spatial location where the action occurred, making it adverbial of place."
4180,omanikeringi,pidama,NaN,adit,Teiseks tulevikueesmärgiks peab Värk strateegilise investori kaasamist panga omanikeringi .,The word 'omanikeringi' refers to ownership or a group of owners and is not a specific location.,"The phrase 'omanikeringi' was classified as 'yes' because it indicates the destination or spatial context related to ownership, qualifying it as an adverbial of place."
5570,karjades,sööma,NaN,in,Need loomad liiguvad ja söövad karjades .,"The phrase 'karjades' describes the movement or grouping of animals, rather than a specific location, so it was classified as 'no'.","The phrase 'karjades' indicates a location or spatial grouping ('in herds'), thus it is classified as adverbial of place."
5860,ettevõttesse,hakkama,NaN,ill,"Kuid enne kui Ferrari 1997. aastal ettevõttesse raha pumpama hakkas , firmal peale kuulsusrikka ajaloo suurt midagi polnud .",The word 'ettevõttesse' refers to entering a company or business and not a physical location.,"The phrase 'ettevõttesse' indicates a spatial movement or location ('into the company'), which qualifies it as an adverbial of place."
9620,klaasist,olema,ära,el,irw rohkem ei maksa aga ... njah käisin hawaii expressis tagumisi piduriklotse vahetamas ja nad lükkasid tule kinnituse nii üles et see nupp millega tuld lahti saada oli vastu sadulat ja sõites kukkus mu tagumine tuli keset pirita teed tagant ära ... õnneks veel töötab kuid klaasist on kild küljest ära ...,"The phrase 'klaasist' refers to 'glass' material and does not point to a location, so it is classified as not location ('no').","The phrase 'klaasist' is an adverbial of place because it describes the origin or source location of the item, which is spatially related."


In [56]:
activity = other1[other1["explanation2"].str.contains("activity")]
activity

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
4751,laulus,hulkuma,NaN,in,"Tõelistele eestlastele , maarahvale , vastandub aga kaasaegses mütoloogias kahtlemata "" kodutu migrant "" , kes hulgub ringi ka ühes Alo Matiiseni populaarses taasärkamisaegses laulus .","The word 'laulus' refers to a song and not a geographical or locational entity, hence it is not classified as a location.","The phrase 'laulus' specifies the location within the song where the activity is happening, classifying it as an adverbial of place."
9570,Mänguseltsis,askeldama,NaN,in,Mänguseltsis askeldanud takuste parukatega eelmise sajandi tegelased mängisid maha ka näitemängujuppe muistsete eestlaste elust-olust .,"The word 'Mänguseltsis' refers to an organization or a social group ('play society') rather than a geographical or locational entity, hence it is not classified as a location.","The phrase 'Mänguseltsis' indicates a specific location where the activity occurred, making it adverbial of place."


In [63]:
program =  other1[(other1["explanation2"].str.contains("TV")) | (other1["explanation2"].str.contains("show")) | (other1["explanation2"].str.contains("program"))]
program

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
531,Saates,vestlema,NaN,in,"Saates "" Naiste jutud "" vestleb reporter Tartu naistekliinikus jõululaste emadega , "" Meeste juttude "" külaliseks on aga Halliste kirikuõpetaja Kalev Raave .",Saates was classified as not location because it refers to a segment or context of a program rather than a geographic location.,The phrase 'Saates' was classified as 'yes' because it indicates a location (in a show or program).
4790,TV-s,jälgima,NaN,in,F1-te jälgis Cooper kuni viimase ajani nii TV-s kui võimalusel koha peal etappidel .,"The word 'TV-s' refers to a medium of communication (television) rather than a physical or geographical location, so it is classified as 'no'.",The phrase 'TV-s' was classified as adverbial of place ('yes') because it indicates the location where Cooper was being followed until the last moment.
5271,põhiõppesse,saama,sisse,ill,"Kuna õppima asuvad välistudengid pole veel selgunud , ei osanud Kääni öelda , kas neist mõni sai põhiõppesse sisse .","The phrase 'põhiõppesse' refers to a specific course of study or program and not to a physical location, so it was classified as 'no'.",The phrase 'põhiõppesse' is classified as an adverbial of place ('yes') because it specifies the destination related to entering the main study program.
6920,ETV-s,olema,alles,in,Kevadel tulevad linnud tagasi ja ETV-s on reklaam alles .,"The phrase 'ETV-s' refers to the Estonian Television channel and not a physical location, hence classified as 'no'.","The phrase 'ETV-s' indicates a location where the event occurs, which makes it an adverbial of place."


In [59]:
other2 = other1[~other1["explanation2"].str.contains('|'.join(["activity", "spatial", "direction","source", "origin", "destination", "specific place", "particular place", "event"]))]
other2

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
200,Rahvusvahelisele,naasma,NaN,all,USA kosmosesüstik Atlantis naasis retkelt Rahvusvahelisele Kosmosejaamale ning maandus kell 09.57 Eesti aja järgi Maal .,The phrase 'Rahvusvahelisele' was classified as not a location because it is an adjective describing 'Kosmosejaamale' (International Space Station) and does not itself specify a place.,'Rahvusvahelisele' was classified as 'yes' because it specifies a location related to the International Space Station.
250,play-offis,edenema,NaN,in,"Kui mõlemad meeskonnad edenevad play-offis , kohtuvad Brasiilia - Prantsusmaa veerand- või poolfinaalis .","The phrase 'play-offis' refers to a competition phase rather than a specific geographic location, so it is not classified as a location ('no').","The phrase 'play-offis' was classified as 'yes' because it indicates the location where the teams progress or compete, making it an adverbial of place."
310,Ekspressis,jagama,NaN,in,"Sest et pandigi fakti ette , "" jagas Reiljan Eesti Ekspressis oma teadmisi Ühtse Venemaa ja Res Publica sidemete kohta .",The phrase 'Ekspressis' was classified as 'no' because it refers to a publication (Eesti Ekspress) and not a physical or geographical location.,"The phrase 'Ekspressis' specifies the location (in 'Eesti Ekspress'), making it an adverbial of place."
470,sisustusajakirjas,kohama,NaN,in,"Iseenesest ju ilus karp , ja kui nüüd hästi järele mõelda , kas polnud ta just selliseid kämpe , seitsmekümnendaid meenutavaid nipsasju hiljuti ühes välismaises sisustusajakirjas kohanud ?",'sisustusajakirjas' was classified as 'no' because it refers to a magazine about interior design rather than a physical location.,The phrase 'sisustusajakirjas' was classified as 'yes' because it describes the location (in an interior design magazine) where something was encountered.
531,Saates,vestlema,NaN,in,"Saates "" Naiste jutud "" vestleb reporter Tartu naistekliinikus jõululaste emadega , "" Meeste juttude "" külaliseks on aga Halliste kirikuõpetaja Kalev Raave .",Saates was classified as not location because it refers to a segment or context of a program rather than a geographic location.,The phrase 'Saates' was classified as 'yes' because it indicates a location (in a show or program).
...,...,...,...,...,...,...,...
9280,Warjekajetasse,nägema,NaN,ill,Sest meije näeme nühd läbbi Warjekajetasse pimmedän Sönnan / ent sis Palgest Palgeni : Nühd tunne minna Tükki wärki / ent sis sah minna tundma / nida kui minnake olle tuttu .,"The term 'Warjekajetasse' does not refer to any recognized geographical location but seems to be a specific or poetic term, so it was classified as 'no'.","The phrase 'Warjekajetasse' indicates the place through which they see the matter, making it adverbial of place."
9560,sõidukitesse,nägema,NaN,ill,Nimelt : trammijuhid ei näe teatud tüüpi sõidukitesse ( KT-4 ehk liigendtramm ) esiuksest sisenejaid .,"The phrase 'sõidukitesse' refers to vehicles in general and does not specify a particular place or location, so it is not classified as a location.","'sõidukitesse' was classified as 'yes' because it indicates location referring to entering into vehicles, which is adverbial of place."
9701,televisioonis,seiklema,NaN,in,"Imemees MacGyver on seigelnud aastate eest Soome televisioonis , kus MTV 3 praegu vanu seeriaid üle kordab .","'televisioonis' is not classified as a location because it refers to a medium (television), not a geographic entity.","The word 'televisioonis' refers to a location (television), so it is classified as an adverbial of place."
9760,Pajuneni,saama,NaN,adit,Läbi Pajuneni sai Ingman ka Spice Ice jäätisekohvikuid haldava OÜ Jäätisefriik omanikuks .,"The phrase 'Pajuneni' refers to a person or entity in this context, not a geographical location.",The phrase 'Pajuneni' is classified as an adverbial of place because it refers to a location associated with how Ingman became the owner.


## mõned selgemad  gpt "no" põhjendused

1. condition /state
2. manner
3. millest
4. event/activity
5. source/origin
6. destination
7. direction
8. spatial
9. not a specific place
10. position (süles, kuklas, roolis, ametis, alguses)
11. movement
12. project, schedule
13. metafor/figurative
14. abstract
15. subject
16. person
17. topic, matter

In [64]:
no_ex = df1[(df1["classification2"]=="no")]
# explanation=esimese korra yes põhjendus, explanation2=teise korra no põhjendus
no_ex = no_ex[["form", "verb", "verb_compound", "morph_case", "sentence", "explanation", "explanation2"]]


In [65]:
no_ex

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
6,ekstaasis,tervitama,NaN,in,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .","The word 'ekstaasis' translates to 'in ecstasy' and refers to an emotional state, not a physical location.","The phrase 'ekstaasis' refers to an emotional state and does not indicate a physical location, so it is not adverbial of place."
8,seisus,ravima,NaN,in,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place."
14,koosseisust,minema,ära,el,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",The phrase 'koosseisust' pertains to organizational structure and not a geographic location.,"The phrase 'koosseisust' is not adverbial of place because it refers to membership or composition, rather than specifying a location."
27,hooldeprojekti,panema,NaN,adit,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .","The phrase 'hooldeprojekti' refers to a project or initiative, not a geographic location, so it is not classified as a location.","The phrase 'hooldeprojekti' was classified as 'no' because it does not indicate a location or answer the question 'where', but rather refers to a project related to care."
32,eelnevasse,ütlema,NaN,ill,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,"The word 'eelnevasse' refers to a preceding context or matter, not a physical or geographical location.","The phrase 'eelnevasse' refers to something previously mentioned or an earlier situation, not indicating a specific place."
...,...,...,...,...,...,...,...
9944,Lennuplaani,liikuma,NaN,adit,"Lennuplaani järgselt Kaliningradi suunas liikunud lennuk sisenes Eesti õhuruumi Vaindloo saare piirkonnas ühe meremiili sügavuselt , viibides Eesti õhuruumis alla minuti .",The phrase 'Lennuplaani' refers to a flight schedule and not a physical location.,"The phrase 'Lennuplaani' refers to a flight schedule, which is a temporal or organizational concept rather than a physical location or place, so it is not adverbial of place."
9946,14-s,käima,NaN,in,"110-st valimisringkonnast 14-s , nende seas ka kolmes pealinna Minski ringkonnas , ei käinud komisjoni väitel oma häält andmas üle poole valijatest ja seal korraldatakse valimiste teine voor .","The phrase '14-s' refers to an ordinal number and not a physical location, hence it was classified as 'no'.","The phrase '14-s' refers to the specific number of constituencies but does not describe a physical location or place, so it is not adverbial of place."
9959,pükstes-pintsakutes,käima,ringi,in,"Et me südametäiega öeldud banaanivabariigi võrdlust tõsiselt ei võta ja käime ringi pükstes-pintsakutes , mitte karunahkades , on loomulik võtta mõõtu oma kultuuriruumist .","The phrase 'pükstes-pintsakutes' refers to clothing and not a location, so it was classified as 'no'.","The phrase 'pükstes-pintsakutes' refers to a manner of dressing rather than a location, so it is not an adverbial of place."
99

In [67]:
condition = no_ex[(no_ex["explanation2"].str.contains("condition")) | (no_ex["explanation2"].str.contains("state"))]
condition

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
6,ekstaasis,tervitama,NaN,in,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .","The word 'ekstaasis' translates to 'in ecstasy' and refers to an emotional state, not a physical location.","The phrase 'ekstaasis' refers to an emotional state and does not indicate a physical location, so it is not adverbial of place."
8,seisus,ravima,NaN,in,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place."
36,langusesse,pöörama,NaN,ill,"Pigem on see madal või pöörab sootuks langusesse . ""","The word 'langusesse' refers to a state or condition (decline) rather than a physical location, so it is classified as 'no'.",The phrase 'langusesse' describes a state or condition (decline) rather than a physical location or place.
202,korras,ravima,NaN,in,"Konsultatsioonid ainult , muidugi ravib ta hobi korras omainimesi .",The phrase 'korras' refers to a state or condition rather than indicating a location.,'korras' was classified as 'no' because it does not indicate a place but rather a state or condition.
374,marutõves,laskma,maha,in,Aprilli alguses laskis üks Kõrveküla mees maha alevikku tunginud marutõves rebase .,The word 'marutõves' refers to a disease (rabies) and not to a specific location.,"The phrase 'marutõves' describes the condition of the fox, not a location, so it is not classified as an adverbial of place."
...,...,...,...,...,...,...,...
9712,kipsis,lamama,NaN,in,"Tragikoomiline videolõik salvestati mullu 22. detsembril Mustamäe haigla palatis , kus KMA osakonnajuhataja Artur Aganitši alluv pärast autoavariid kipsis jalaga lamas .","The word 'kipsis' describes a condition or state (in a cast), not a geographical location.","The phrase 'kipsis' describes a state (being in a cast) and not a location, so it is not adverbial of place."
9868,härmaehtes,rippuma,NaN,in,Nädalate kaupa rippusid maja ees kasvavad kased raskes härmaehtes .,"The word 'härmaehtes' refers to being covered in frost and not a geographical location, so it was classified as not location ('no').",The phrase 'härmaehtes' is not an adverbial of place because it describes a state or condition rather than a location.
9872,mõistmatuses,laiutama,NaN,in,"“ Ma laiutan mõistmatuses käsi haridusametnike ja lapsevanemate ees , ma ei ole don Quiote , et tuuleveskitega võidelda , vaid jalutan minema , ” kõlas Dan Põldroosi napp kommentaar .","The phrase 'mõistmatuses' describes a state of being, not a geographical location, so it is not classified as a location.","The phrase 'mõistmatuses' is not adverbial of place because it describes a state of confusion or misunderstanding, not a specific location."
9883,häälde,tungima,NaN,adit,» Pisarad tungisid reetlikult mu häälde .,"The phrase 'häälde' refers to a vocal expression and not a physical place; thus, it is not a location.","The phrase 'häälde' is not an adverbial of place because it describes the manner or condition related to the voice, not a location."


In [68]:
manner = no_ex[(no_ex["explanation2"].str.contains("manner"))]
manner

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
190,seelikutes,juhatama,NaN,in,Sajandi viimasel kümnendil juhatavad triibulistes seelikutes lõbusad mehikesed eesti spordipubliku taas Kadriorgu jalgpalli vaatama .,"'seelikutes' is not classified as a location because it refers to skirts, a type of clothing, and is not related to a geographical location.",The phrase 'seelikutes' is not adverbial of place because it describes the manner of attire rather than indicating a location.
242,korras,laskma,maha,in,"Võib-olla suri selle tüübi vanaema kusagil Siberis ja tema isa ei võetud ülikooli , sest ta oli rahvavaenlaste laps , aga tema vanaema vend kükitas kusagil Võrumaa metsas , kuni ta lasi haarangu korras maha tema oma sõber metsavend , kes osutus KGB nuhiks .","The phrase 'korras' refers to 'in order' or 'arranged' and not a location, so it is classified as 'no'.","The phrase 'korras' describes the manner in which the action occurred and does not indicate a specific location, so it is not an adverbial of place."
248,neorenessanss-stiilis,ehitama,NaN,in,1864. aastal omandas aiakrundi raamatukaupmees Karow ja ehitas siia neorenessanss-stiilis elumaja .,The phrase 'neorenessanss-stiilis' was classified as 'no' because it describes an architectural style and not a location.,"The phrase 'neorenessanss-stiilis' describes a style or manner of construction, not a location, so it is not an adverbial of place."
253,punkt-punktilt,kulgema,NaN,abl,"Mäng püsis Hotronicul ohjes algusest peale , kuigi enamjagu mänguajast kulges heitlus punkt-punktilt .","The phrase 'punkt-punktilt' refers to a gradual or step-by-step manner, not a physical location.","The phrase 'punkt-punktilt' was classified as 'no' because it describes the manner of the competition, not the location where it occurred, and is therefore not an adverbial of place."
428,tõusujoones,edenema,NaN,in,"Revolutsioon edenes lainetena tõusujoones , saavutades haripunkti 1905. aasta sügisel , kui riiki haaras poliitiline üldstreik .","The phrase 'tõusujoones' was classified as 'no' because it describes a metaphorical progression, not a physical location.","The phrase 'tõusujoones' describes a manner or progression, not a specific place, so it is not an adverbial of place."
...,...,...,...,...,...,...,...
9753,ebakainelt,sõitma,NaN,abl,"Ebakainena on roolis olnud 37 protsenti juhtidest , rohkem kui üks protsent kõigist juhtidest sõidab aga pidevalt ebakainelt .","The phrase 'ebakainelt' was classified as 'no' because it describes a condition or manner (intoxicated), not a location.","The phrase 'ebakainelt' is classified as 'no' because it describes the manner of driving (intoxicated), not the location where the driving occurred."
9807,diagonaalis,sadama,NaN,in,"Vihm , räägitakse , sajab Islandil tuule tõttu peaaegu alati diagonaalis .","The word 'diagonaalis' refers to a diagonal direction, describing an orientation rather than a physical or geographical location.","The word 'diagonaalis' describes the manner or direction of the rain, not a specific place, so it was classified as 'no'."
9883,häälde,tungima,NaN,adit,» Pisarad tungisid reetlikult mu häälde .,"The phrase 'häälde' refers to a vocal expression and not a physical place; thus, it is not a location.","The phrase 'häälde' is not an adverbial of place because it describes the manner or condition related to the voice, not a location."
9959,pükstes-pintsakutes,käima,ringi,in,"Et me südametäiega öeldud banaanivabariigi võrdlust tõsiselt ei võta ja käime ringi pükstes-pintsakutes , mitte karunahkades , on loomulik võtta mõõtu oma kultuuriruumist .","The phrase 'pükstes-pintsakutes' refers to clothing and not a location, so it was classified as 'no'.","The phrase 'pükstes-pintsakutes' refers to a manner of dressing rather than a location, so it is not an adverbial of place."


In [71]:
event = no_ex[(no_ex["explanation2"].str.contains("event")) | (no_ex["explanation2"].str.contains("activity"))]
event

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
320,MM-i,kandma,NaN,adit,Sporti on muidugi enamgi : rahvustelevisioon kannab üle kergejõustiku MM-i Helsingist ja sõudmise MM-i ning erisaated kajastavad Tour de France'i .,"The term 'MM-i' (likely short for 'world championship') refers to an event or competition, not a specific place.",The phrase 'MM-i' was classified as 'no' because it refers to a specific event or competition (world championships) and not a location or place.
452,protsessides,ringlema,NaN,in,"Raske on veekogust välja püüda seal massiliselt kasvavaid taimekogumeid ( lemled , kardhein ) , aga sügise poole hakkavad nad lagunema , andes lisa biogeenidele , mis ringlevad veekogusisestes protsessides .","'protsessides' refers to processes or activities, not a location.","The phrase 'protsessides' is not an adverbial of place because it indicates a location within a process or activity, not a physical place."
538,milles,elama,kaasa,in,"» Tippmargi 7.45 hüppas Drechsler 1986. aasta juunis NSV Liidu ja Saksa DV matšil , milles Eesti publik pigem sakslastele kaasa elas .","The phrase 'milles' is a relative pronoun referring to an unspecified event or context, not a location, so it is classified as 'no'.",The phrase 'milles' was classified as 'no' because it refers to a situation or event and not a physical location.
551,USA-visiiti,pidama,NaN,adit,Poliitikud peavad kaitseministri USA-visiiti Eestit häbistavaks,"The phrase 'USA-visiiti' refers to a visit, not the location itself, so it is classified as 'no'.","The phrase 'USA-visiiti' refers to the act of a visit to the USA, which is an event rather than a specific location, so it is not classified as an adverbial of place."
564,keraveeretamisele,jooksma,NaN,all,"Aga mis see pisike Küpros nii imelist korda on saatnud , et saareelanikud keraveeretamisele tormi jooksevad ! ?","The phrase 'keraveeretamisele' refers to an action and not a specific geographical or physical location, so it is classified as 'no'.",The phrase 'keraveeretamisele' was classified as 'no' because it indicates an activity or event rather than a location.
899,maamõõtmisse,laienema,NaN,adit,"Laienes maamõõtmisse , kinnisvarasse , autode müüki ja remonti ...","The phrase 'maamõõtmisse' refers to an activity or field of work, not a geographical location, so it was classified as 'no'.","The phrase 'maamõõtmisse' indicates an area of engagement or activity, not a specific location, so it is not an adverbial of place."
1264,läbikäima,pidama,NaN,adit,"Aga programmi põhi olemus on selles , et programm peab läbikäima kõik joonises olevad elemendid ja kontrollime nende propertie ( kiht värv stiil joonejämedus ) õigsust .",The phrase 'läbikäima' refers to an action and not a location.,The phrase 'läbikäima' was classified as 'no' because it denotes an action or activity rather than a location or place.
1425,meigikollektsioonis,võimutsema,NaN,in,Revloni kevadises meigikollektsioonis LavenDare võimutseb lilla .,"The word 'meigikollektsioonis' refers to a makeup collection, which is a category or product and not a physical place.","Classified as 'no' because 'meigikollektsioonis' indicates the setting or context of the event, not the location in a spatial sense."
1476,hiigelavariis,põrkama,kokku,in,"Moskva-Riia maanteel põrkasid täna hommikul mitmes hiigelavariis kokku 40 sõidukit , inimesed vigastada ei saanud .","The phrase 'hiigelavariis' describes an event (a large-scale accident), not a physical location, so it is classified as 'no'.","The phrase 'hiigelavariis' refers to an event rather than a specific location, so it is not an adverbial of place."
1499,maailmasõjas,maabuma,NaN,in,1944 - Liitlasväed maabusid Teises maailmasõjas Hollandi Uus-Guineas .,"The term 'maailmasõjas' refers to an event (world war), not a physical location.","It was classified as 'no' because 'maailmasõjas' specifies an event, not a place."


In [72]:
source = no_ex[(no_ex["explanation2"].str.contains("source")) | (no_ex["explanation2"].str.contains("origin"))]
source

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
37,lisarahast,jõudma,tagasi,el,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .","The phrase 'lisarahast' refers to additional funding and does not denote a location, hence classified as 'no'.","The phrase 'lisarahast' refers to additional funding or resources, not specifying a place."
448,millest,tulema,ära,el,"Pärast finišit ilmnes ka põhjus veritsevad varbad ja sinakaks tõmbunud varbaküüned , millest lõpuks seitse ära tulid .","The word 'millest' functions as a pronoun and does not indicate a location, so it was classified as 'no'.",The phrase 'millest' was classified as 'no' because it does not describe a location but instead refers to the cause or source of something.
1134,narvakas,üürima,NaN,in,Kinnitamata andmetel oli tapetud ekspolitseinikust narvakas Andrei üürinud Kohtla-Järvel korteri ning asunud seal koos kolme naisterahvaga Narvast toodavat moonipuru müüma .,"The word 'narvakas' refers to a person from Narva, which is a characteristic, not a specific place referring to a location.","The phrase 'narvakas' refers to a person from Narva, which is not an adverbial of place but a noun describing a person's origin."
1484,1kesklinnast,sõitma,NaN,el,Helsingis Kuusilahdenkuja 1kesklinnast sõidavad sinna bussid nr 194 ja 195 platvormilt 50.,The phrase '1kesklinnast' refers to a starting point but does not denote a specific geographic location.,"The phrase '1kesklinnast' refers to the starting point of a route or origin, not the specific place where an action is happening, so it is not adverbial of place."
1493,toodangust,importima,NaN,el,"Ettevõte importis ligi poole toodangust Inglismaale , Norrasse , Rootsi ja Soome .","The phrase 'toodangust' refers to production, which is not a location.","It was classified as 'no' because 'toodangust' refers to the source of the imported goods, not indicating a specific location."
1780,kaasadest,tellima,NaN,el,"Näitusel on väljas Hannes Tamjärve ja Jüri Mõisa portreed , kuuldavasti tellivad noored ärimehed Okaselt akte oma kaasadest .","The phrase 'kaasadest' refers to individuals or people, not a location, so it was classified as 'no'.",The phrase 'kaasadest' is not an adverbial of place because it indicates the origin or participants rather than a location.
1787,draamateosest,kostma,NaN,el,"Kui pähesööbinud repliigid tuntud draamateosest saalist kostavadki , siis vaid selleks , et anda publikule väike vihje , mis teema ja motiiviga parasjagu tegu on .",The word 'draamateosest' refers to a literary work (a drama) and not a physical location.,"The phrase 'draamateosest' is not an adverbial of place because it indicates the source of content, not a location."
1834,hobustelt,kukkuma,NaN,abl,"Kajamaal Steeplechase Business Cup 2002-l harrastajate sõidul hobustelt kukkunud Evelin Evisalu ( 22 ) lebab Mustamäe haigla palatis , vaatab telekat ja meenutab ohtlikku võistlust .","The word 'hobustelt' refers to 'horses' in the context of falling off, not a specific location, so it is classified as 'no'.","The phrase 'hobustelt' refers to the source or origin (from the horses) rather than specifying a location, so it is not adverbial of place."
1849,teatel,lamama,NaN,ad,Ta lamab BBC teatel päevad läbi voodis ega suuda ilma kõrvalise abita kõndida .,"The phrase 'teatel' means 'according to' and refers to a source of information, not a physical location.","The phrase 'teatel' is not adverbial of place because it refers to a source or means of information, not a location."
1889,intervjuust,avastama,NaN,el,"Arvasin , et konsumerism ja rahavalitsus teevad haiget , aga siis avastasin teie intervjuust SL Õhtulehele , et tegelikult lausa ootate aega , mil riigikogulase suurt pensioni saama hakkate , et saaks oma maamajja asju osta .",The word 'intervjuust' is not a location; it refers to an interview and not a specific place.,"The phrase 'intervjuust' refers 

In [73]:
dest = no_ex[(no_ex["explanation2"].str.contains("destina"))]
dest

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
1785,dokumentidesse,leidma,NaN,ill,Tolliametnikud leidsid koormast aga peale dokumentidesse märgitu veel ka 20 Toshiba telerit .,"The phrase 'dokumentidesse' was classified as 'no' because it refers to documents, not a location.","The phrase 'dokumentidesse' is not an adverbial of place because it refers to the destination of an action into documents, not a physical or defined location."
4298,imterpreetorisse,olema,NaN,ill,"või imterpreetorisse - vahet pole , kuid arvasin , et see lahendus on liialt tavaline ...","The phrase 'imterpreetorisse' does not denote a physical location but refers to a tool or concept, so it was classified as 'no'.","The phrase 'imterpreetorisse' refers to a direction or destination but does not specify a physical location, so it was classified as 'no'."
5187,eelarvesse,tahtma,NaN,ill,"Riik tahab eelarvesse tulude leidmiseks müüa avalikul pakkumisel Toompeal Rahukohtu 3 asuva hoone , millest varem pidi saama uus valitsuse residents .","The phrase 'eelarvesse' refers to the budget and not a geographical location, hence it was classified as 'no'.","The phrase 'eelarvesse' does not indicate a physical location but rather a figurative destination in a budgetary context, so it is not an adverbial of place."


In [74]:
no_ex[(no_ex["explanation2"].str.contains("millest"))]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
448,millest,tulema,ära,el,"Pärast finišit ilmnes ka põhjus veritsevad varbad ja sinakaks tõmbunud varbaküüned , millest lõpuks seitse ära tulid .","The word 'millest' functions as a pronoun and does not indicate a location, so it was classified as 'no'.",The phrase 'millest' was classified as 'no' because it does not describe a location but instead refers to the cause or source of something.
6722,millest,tõstma,välja,el,"Häärberi parema tiiva ukse lähedale sõidab kaubik , millest kelnerid ja kokad tõstavad kümmekond minutit välja kandameid toiduga .",The phrase 'millest' is a relative pronoun meaning 'from which' and does not denote a physical or geographic location.,"The phrase 'millest' was classified as 'no' because it refers to a means (the van from which objects are taken), not a place."
7231,millest,jõudma,tagasi,el,""" Tellisime Eestist viis 40 istekohaga väikelennukit AN 72 , millest kaks lennukit 80 inimesega jõudis eile õhtuks Eestisse tagasi .",The phrase 'millest' refers to a partitive form and does not indicate a physical or geographical location.,The phrase 'millest' specifies part of something but does not indicate a place.
7425,millest,importima,NaN,el,"Latvenergo on täielikult Läti riigile kuuluv energeetikaettevõte , mis müüs 1999. aastal 6,0 TWh elektrienergiat , millest 2,0 TWh importis teistest riikidest .","The phrase 'millest' refers to a portion or part, not a location.","The phrase 'millest' refers to a part of something but does not indicate a physical or conceptual location, so it is not an adverbial of place."
9982,millest,rändama,NaN,el,"17. jaanuaril avatakse Tallinna Kunstihoones John Smithi järjekordne suur ühisnäitus , millest märkimisväärne osa rändabki suve hakul Veneetsiasse .","The phrase 'millest' does not refer to a specific geographic or physical location; instead, it relates to a part of the exhibition that will be moved.","The phrase 'millest' refers to a part of the exhibition, not a physical location, so it is not an adverbial of place."


In [75]:
direct = no_ex[(no_ex["explanation2"].str.contains("direction"))]
direct

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
506,kaitsele,kerkima,NaN,all,Nagu ikka kerkivad oma lemmiku kaitsele muusikaprofessori ustavad jüngrid .,"'kaitsele' refers to the act of protecting, not a physical or geographical place.","The phrase 'kaitsele' was classified as not adverbial of place ('no') because it describes intent or direction related to defending someone, rather than a specific physical location."
621,poolelt,väljuma,NaN,abl,"Väljume kaubamaja teiselt poolelt Kaubamaja tänaval , sealt väljumine on sisenemisest tunduvalt lihtsam ning Indrek saab sellega hakkama kõrvalise abita .","The phrase 'poolelt' describes a directional position rather than a specific physical location, so it is not classified as a location.","The phrase 'poolelt' in the sentence does not indicate a specific location or direction, so it is not classified as an adverbial of place."
841,suunda,avanema,NaN,adit,"Näiteks Kassitoomel avaneb ilusaid vaateid igasse suunda , kuid ühtki pinki istumiseks seal ei ole .",NaN,"The phrase 'suunda' refers to a direction, which is not a specific physical location or spatial context, so it is not considered an adverbial of place."
1020,suunas,juhatama,NaN,in,"Haigete jalgade tõttu suutis ta vahemaast läbida vaid kolmandiku , kuid juhatas poisse õiges suunas edasi jooksma .",The phrase 'suunas' was classified as 'no' because it indicates a direction (towards something) but does not specify a concrete geographic or physical location.,The phrase 'suunas' was classified as 'no' because it indicates direction or purpose rather than a specific location.
1243,arendamisse,tahtma,NaN,adit,"neljatakti tehnoloogia on vorreldes kahe taktiga nii heaks arenenud , kuna kahe takti arendamisse ei taha keegi enam raha matta , sest olemasolevate tehniliste lahendustega ( kolviga juhitav gaaside liikumine ) ei saa kahte takti okonoomselt toole panna , seega ei ole seda eriti voimalik muua .","The word 'arendamisse' refers to the act of developing or investing in development, not a specific location.","The phrase 'arendamisse' describes a purpose or direction (related to development) rather than indicating a specific location, and is thus not classified as an adverbial of place."
1725,Otsa,kihutama,NaN,adit,"Otsa kihutanud auto tõttu kolmapäeva hilisõhtul Lasnamäel teelt välja vastu majaseina paiskunud tramm naaseb rööbastele uuel aastal , sest enne juurdluse lõppu ei julge trammipark ühissõidukit oma raha eest remontima hakata .",The phrase 'Otsa' refers to an action or direction but does not specify a location.,The phrase 'Otsa' was classified as 'no' because it does not refer to a specific location or where something happens but rather suggests a direction or movement towards an end.
1938,külgedelt,ulatuma,NaN,abl,"Külmik , pliit , kraanikauss , praeahjud ja tööpinnad on integreeritud ühtsesse kapiseina , mis ulatub külgedelt ettepoole ja taandub nõgusalt keskosas .","The phrase 'külgedelt' means 'from the sides' in English, which indicates direction or position rather than a specific location.","The phrase 'külgedelt' refers to a direction or origin but not a specific physical location, so it is not an adverbial of place."
2385,meestesse,hakkama,NaN,ill,"Tütardest kujunevad ajapikku emade koopiad , nii võis arvata , et Ööstki kasvab sinisukk , kes meestesse üldse ja papasse eriti hakkab suhtuma patroneeriva ükskõiksusega .",?,"The phrase 'meestesse' was classified as 'no' because it indicates a direction of attitude or relation rather than a physical location, and thus is not an adverbial of place."
2477,suundadesse,kaduma,NaN,ill,Paraku kadus see raha hoopis teistesse suundadesse .,"The term 'suundadesse' means 'directions' and is used in a metaphorical sense here, not referring to a physical location, hence classified as 'no'.","The phrase 'suundadesse' refers to directions or tendencies rather than specific locations, so it was classified as 'no'."
2738,koha-likku,pöörduma,NaN,adit,"Kui olete j

In [77]:
spatial = no_ex[(no_ex["explanation2"].str.contains("spatial"))]
spatial

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
73,Risbiteri,saama,NaN,adit,[ Saab Risbiteri mõõga kätte .,The phrase 'Risbiteri' was classified as 'no' because it appears to refer to a specific object (possibly a sword) rather than a location.,The phrase 'Risbiteri' was not classified as an adverbial of place because it refers to a proper noun (likely a name) rather than a location or spatial context.
78,alguses,põrkama,kokku,in,Viimase ringi alguses põrkas Tobreluts raja lahknemiskohas sloveenlasega kokku .,"'alguses' refers to the timing of an event, not a specific location.",The phrase 'alguses' was not classified as an adverbial of place because it refers to a temporal context (the beginning) rather than a spatial one.
176,rahmeldamisest,hankima,NaN,el,"« Erki natuur nõuab pidevat tegutsemist , võimalik , et ta hangib rahmeldamisest energiat . »","The word 'rahmeldamisest' refers to an action or behavior rather than a specific location, so it was classified as 'no'.","The phrase 'rahmeldamisest' is not about a location or spatial reference, so it is not classified as an adverbial of place."
380,adresse,ütlema,NaN,ill,asi: raffas öelge oma MSN adresse,"The phrase 'adresse' refers to addresses in a general sense and not to a specific location, hence it is classified as 'no'.",The phrase 'adresse' is not adverbial of place because it does not indicate a location or spatial relation.
841,suunda,avanema,NaN,adit,"Näiteks Kassitoomel avaneb ilusaid vaateid igasse suunda , kuid ühtki pinki istumiseks seal ei ole .",NaN,"The phrase 'suunda' refers to a direction, which is not a specific physical location or spatial context, so it is not considered an adverbial of place."
...,...,...,...,...,...,...,...
9621,ainest,tellima,NaN,el,"Sest siiani tellis Tiit Vähi oma kiirgavat ainest Venemaalt läbi Narva ja sealsed ametnikud olid juba harjunud - ületab jah lubatud näite , aga eks Vähi ise teab , millega riskib .","The phrase 'ainest' refers to a substance or material and does not indicate a specific place or location, so it was classified as 'no'.",The phrase 'ainest' is not an adverbial of place because it indicates material or substance rather than a location or spatial reference.
9627,reaalkasvu,lubama,NaN,adit,"Briti tervishoiukulutused on hetkel allpool Euroopa keskmist taset , rahandusminister Gordon Brown lubas mullu siiski tervishoiukulude reaalkasvu 6,1 protsenti aastas kuni aastani 2001.","The phrase 'reaalkasvu' was classified as 'no' because it refers to a concept of growth rate, not a physical location.",The phrase 'reaalkasvu' is not an adverbial of place because it pertains to economic growth rather than a spatial aspect or physical location.
9632,A-s,võistlema,NaN,in,Karjuse sõnul võistleb Intercontinental-C klassile mainekuselt järgnevas vormel Super A-s juba 15 sportlast .,"The phrase 'A-s' refers to a classification or category, not a physical location, so it is classified as 'no'.","The phrase 'A-s' refers to a class or category, not a physical or spatial location."
9734,dzhässikõladest,kostma,NaN,el,"Soomest , Rootsist , Venemaalt ja Fääri saartelt jõuab Tallinna muusika , mille dzhässikõladest kostab ka rocki ja etnot .",The word 'dzhässikõladest' was classified as 'no' because it refers to musical sounds and not a physical location.,The phrase 'dzhässikõladest' was classified as 'no' because it does not indicate a location or spatial reference; it instead describes the source of music sounds.


In [78]:
place = no_ex[(no_ex["explanation2"].str.contains("specific place")) | (no_ex["explanation2"].str.contains("particular place"))]
place

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
32,eelnevasse,ütlema,NaN,ill,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,"The word 'eelnevasse' refers to a preceding context or matter, not a physical or geographical location.","The phrase 'eelnevasse' refers to something previously mentioned or an earlier situation, not indicating a specific place."
102,eelarvesse,olema,NaN,ill,"Tallinna rahandusküsimustega tegelev abilinnapea Ants Leemets kinnitas esmaspäeva õhtul oma kabinetis “ Postimehele ” selge sõnaga , et vähemalt selleks aastaks on Nomura laenu tagasimaksmiseks linnal raha juba olemas , s.o sisse planeeritud eelarvesse , mille volikogu nii või teisiti peagi vastu võtab .",The phrase 'eelarvesse' refers to a budget rather than a geographical or physical location.,The phrase 'eelarvesse' was classified as 'no' because it indicates a target or goal (to the budget) rather than a specific place.
428,tõusujoones,edenema,NaN,in,"Revolutsioon edenes lainetena tõusujoones , saavutades haripunkti 1905. aasta sügisel , kui riiki haaras poliitiline üldstreik .","The phrase 'tõusujoones' was classified as 'no' because it describes a metaphorical progression, not a physical location.","The phrase 'tõusujoones' describes a manner or progression, not a specific place, so it is not an adverbial of place."
586,väsimusest,tulema,ära,el,"Au neile , et väsimusest hoolimata võit ära tuli , » kiitis A.Le Coqi peatreener Heino Enden .",The word 'väsimusest' refers to the state of fatigue and does not imply a location.,"The phrase 'väsimusest' refers to a state of fatigue rather than indicating a specific place, so it is not an adverbial of place."
764,taimelehtedesse,kasutama,NaN,ill,"Salatipiigad ja Brokolipoisid on PETA egiidi all tegutsevad rühmitused , kes kasutavad taimelehtedesse riietumist taimetoitluse propageerimiseks .",The word 'taimelehtedesse' refers to leaves and not to a specific location.,The phrase 'taimelehtedesse' was classified as 'no' because it describes what the group is wearing rather than indicating a specific place.
880,tuledes,kõrguma,NaN,in,"Ja siis olime ühes hotellitoas kõrgel korrusel , kõrgusime üle tuledes linna ja nikkusime nagu nugised .","The word 'tuledes' refers to being lit or illuminated, which describes a state rather than indicating a specific location.",The phrase 'tuledes' describes a feature or state (in lights) and does not indicate a specific place.
1039,sõltuvusse,pöörduma,tagasi,adit,"Kas Te võtate omaks , et Teie peaministriks oleku ajal on RAS Silmet pöördunud läänemaailmaga koostöö otsimise teelt tagasi täielikku sõltuvusse Venemaast ?","The phrase 'sõltuvusse' denotes a state of dependency and not a geographical place, so it is classified as 'no'.",The phrase 'sõltuvusse' was classified as 'no' because it indicates a state of dependence rather than a specific place or location.
1484,1kesklinnast,sõitma,NaN,el,Helsingis Kuusilahdenkuja 1kesklinnast sõidavad sinna bussid nr 194 ja 195 platvormilt 50.,The phrase '1kesklinnast' refers to a starting point but does not denote a specific geographic location.,"The phrase '1kesklinnast' refers to the starting point of a route or origin, not the specific place where an action is happening, so it is not adverbial of place."
1646,liikumis,kõndima,NaN,in,"Kui aga seest tulistada , siis Kuul lendab edasi , mitte ei kuku , sest kui lennuk on õhus ja lendab üpris kiiresti ( muidugi mitte püssi kuuli kiirusel ) siis sa saad kõndida seal lennukis ja kui need kiirused oleksid sinu liikumis kiirusega samad siis sa ikka kõnniksid lennukis ja nii ka kuul lendaks .","The word 'liikumis' relates to movement, which does not specify a location.","The phrase 'liikumis' was classified as 'no' because it pertains to movement or motion, not a specific place."

In [82]:
position = no_ex[(no_ex["explanation2"].str.contains("position"))]
position

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
14,koosseisust,minema,ära,el,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",The phrase 'koosseisust' pertains to organizational structure and not a geographic location.,"The phrase 'koosseisust' is not adverbial of place because it refers to membership or composition, rather than specifying a location."
382,kümnes,passima,NaN,in,"Tundub , et nii ETV kui ka TV3 tegijatel , kelle saated teises või kolmandas kümnes passivad ja siis korraga kõigile üllatuseks mõnel nädalal edetabelisse äkksööste teevad , jääb üle paluda kõigevägevamat , et sellised sööstud taas nende kõrvu ja silmi paitaksid .",The word 'kümnes' refers to a numerical order or ranking and not a physical location.,The phrase 'kümnes' is not adverbial of place because it refers to a numerical position or order rather than a physical location.
572,Korvpalli,viskama,NaN,adit,Korvpalli Suproliigas viskas Gert Kullamäe lõpuminutil Oostende Telinduse seitse viimast punkti ja tüüris koduklubi 83 : 77 võiduni Saloniki Iraklise vastu .,"The phrase 'Korvpalli' was classified as 'no' since it refers to basketball, which is not a location but a type of sport.","It was classified as 'no' because 'Korvpalli' refers to basketball, and it does not indicate a specific location or position, which is required for an adverbial of place."
595,kadmist,avastama,NaN,el,"Ise pärast aastaid erinevate saabaste ( USA , Hiina , Vene ) kadmist avastasin ootamatult , et oma Alpi on nendest ikka peajagu üle .",The phrase 'kadmist' does not refer to a specific geographic or locational entity; it generally means 'searching' or 'seeking'.,The phrase 'kadmist' was classified as 'no' because it does not indicate a specific location or position—its focus is on an action or process and not a place.
637,eesotsas,ehitama,NaN,in,Pärnu linnaarhitekt Olev Siinmaa hülgas just umbes sel ajal oma senise traditsionalismi ning ehitas 1930. aastatel Pärnusse Eesti fungi säravad pärlid eesotsas omaenda eramuga .,"The phrase 'eesotsas' means 'at the forefront', which is not a physical or geographical location, so it was classified as 'no'.","The phrase 'eesotsas' refers to a position or role within a group, not a physical location, so it is not an adverbial of place."
874,investoritesse,olema,NaN,ill,"Täna on siiski kindlustunne kodumaistesse investoritesse tagasi tulemas , samuti on juba saadud kasumeid ning ollakse valmis ka uuesti investeerima .","The phrase 'investoritesse' refers to investors, which is not a location, so it was classified as not a location ('no').","The phrase 'investoritesse' refers to investors and does not describe a location or position, so it is not adverbial of place."
1058,15sse,saama,NaN,ill,Maaliit sai 15sse ja Isamaaliit 14sse volikokku .,"The phrase '15sse' represents a numerical or positional reference rather than a location, so it was classified as 'no'.","The phrase '15sse' indicates a numerical position or ranking rather than a location, so it is not an adverbial of place."
1060,süles,kõndima,ringi,in,"Isegi siis , kui poisid oli tited ja ärkasid öösel üles , oli just tema see , kes , laps süles , toas ringi kõndis ja nad niimoodi magama kussutas . ""","The phrase 'süles' was classified as not a location because it refers to the position of holding something in one's lap, which is not a physical location or place.","The phrase 'süles' describes a position or manner of holding the child, not a location where the action takes plac

In [84]:
move = no_ex[(no_ex["explanation2"].str.contains("movement"))]
move

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
98,peitu,pagema,NaN,adit,"Valdo Lips lõi tempo alla , Ivo Saksakulm ja Chris Moore kontrollisid lauavõitlust , Indrek Varblane ja Andrus Nagel pagesid kaitse haardest peitu ja tulistasid kaugvisetega .","The phrase 'peitu' was classified as 'no' because it describes a state or action of hiding, not a location.",The phrase 'peitu' is not classified as an adverbial of place because it indicates a movement to hide rather than specifying a precise location.
1646,liikumis,kõndima,NaN,in,"Kui aga seest tulistada , siis Kuul lendab edasi , mitte ei kuku , sest kui lennuk on õhus ja lendab üpris kiiresti ( muidugi mitte püssi kuuli kiirusel ) siis sa saad kõndida seal lennukis ja kui need kiirused oleksid sinu liikumis kiirusega samad siis sa ikka kõnniksid lennukis ja nii ka kuul lendaks .","The word 'liikumis' relates to movement, which does not specify a location.","The phrase 'liikumis' was classified as 'no' because it pertains to movement or motion, not a specific place."
1725,Otsa,kihutama,NaN,adit,"Otsa kihutanud auto tõttu kolmapäeva hilisõhtul Lasnamäel teelt välja vastu majaseina paiskunud tramm naaseb rööbastele uuel aastal , sest enne juurdluse lõppu ei julge trammipark ühissõidukit oma raha eest remontima hakata .",The phrase 'Otsa' refers to an action or direction but does not specify a location.,The phrase 'Otsa' was classified as 'no' because it does not refer to a specific location or where something happens but rather suggests a direction or movement towards an end.
2400,kokku-lahku,liikuma,NaN,adit,Dekoratsiooni aset täitev nelja mängupinnaga kaldteedest ja treppidest koosnev moodustis liikus kokku-lahku .,"The word 'kokku-lahku' refers to a movement or action, not a physical or geographic location.","The phrase 'kokku-lahku' describes a manner of movement or action, not a location, so it is not adverbial of place."
2804,lendu,liikuma,NaN,adit,"Vana väsind sepa valge päänutt akne all uinakut tegemas , juuksed liiguvad lendu nigu võililleudsu jälle . ”","The word 'lendu' refers to flight or motion and not a geographic or physical place, hence it is not a location.","The phrase 'lendu' (into flight) describes movement but not a specific location, so it is not an adverbial of place."
6351,luusse,laskma,NaN,ill,Laseb leiba luusse oma bussis,"The word 'luusse' refers to resting or relaxation metaphorically and does not denote a geographical or physical location, so it is not classified as a location.","The phrase 'luusse' does not specify a location; it denotes movement but not a clear place, so it is not an adverbial of place."
7044,tehnoloogiasse,moodustama,NaN,ill,"• Hoone ehituse projekti kogumaksumus on 95 miljonit krooni , millest 11 miljonit moodustavad investeeringud tehnoloogiasse , sealhulgas kontsertorelisse .","The word 'tehnoloogiasse' refers to technology and not a physical place, so it is not classified as a location.","The phrase 'tehnoloogiasse' indicates movement into technology and does not specify a location, so it is not an adverbial of place."
7045,nishshidesse,kukkuma,NaN,ill,"Samavõrra ideoloogiliste nishshidega on uue poliitika otsingute juures olulised elu nishshidesse kukkunud tegelased , kes omasuguseid rahulolematuid koondada suudaksid .",?,"The phrase 'nishshidesse' indicates movement into niches, which is metaphorical and not a specific physical location, so it is not an adverbial of place."
7332,tuules,sõitma,ringi,in,"Teise 16,7 km pikkuse ringi sõitsite Mihhail Botvinovi tuules .","tuules was classified as 'no' because it refers to being in the figurative or physical flow of air, not a specific location.","The phrase 'tuules' describes the condition or manner of Mihhail Botvinovi's movement rather than a specific place, thus it is not an adverbial of place."
7850,käest,rändama,NaN,el,nimel rändasid käitised omade vahel käest kätte .,"The phrase 'käest' does not denote a specific geographic or physical place, so it is 

In [86]:
no_ex[(no_ex["explanation2"].str.contains("project")) | (no_ex["explanation2"].str.contains("schedule"))]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
27,hooldeprojekti,panema,NaN,adit,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .","The phrase 'hooldeprojekti' refers to a project or initiative, not a geographic location, so it is not classified as a location.","The phrase 'hooldeprojekti' was classified as 'no' because it does not indicate a location or answer the question 'where', but rather refers to a project related to care."
5385,saatekavas,laiutama,NaN,in,"Ometi oldi ülekandeks ju valmis , sest saatekavas laiutas peo tarvis kolmetunnine auk .","The phrase 'saatekavas' refers to a schedule or program, not to a specific location.","The phrase 'saatekavas' refers to something in the program schedule, which does not denote a specific location or physical place, so it is not an adverbial of place."
6637,kavas,tervitama,NaN,in,Esmakordselt seni ainult naistele mõeldud Maijooksu seitsmeteistkümneaastases ajaloos kavas olnud Meestejooksu tervitasid rõõmuga nii osavõtjad kui pealtvaatajad kahesaja meesjooksja vahel läks tõeliseks rebimiseks .,"The phrase 'kavas' refers to being included in a program or schedule, not a specific location.","The phrase 'kavas' was classified as 'no' because it refers to a program or schedule and not a physical location, so it does not indicate a place."
9944,Lennuplaani,liikuma,NaN,adit,"Lennuplaani järgselt Kaliningradi suunas liikunud lennuk sisenes Eesti õhuruumi Vaindloo saare piirkonnas ühe meremiili sügavuselt , viibides Eesti õhuruumis alla minuti .",The phrase 'Lennuplaani' refers to a flight schedule and not a physical location.,"The phrase 'Lennuplaani' refers to a flight schedule, which is a temporal or organizational concept rather than a physical location or place, so it is not adverbial of place."


In [87]:
no_ex[(no_ex["explanation2"].str.contains("figurative")) | (no_ex["explanation2"].str.contains("metaphor"))]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
54,kraesse,toimuma,NaN,ill,Raudteelaste kraesse toimunud õnnetusi Rentiku sõnul siiski veeretada ei saa .,"The phrase 'kraesse' does not indicate a location; it metaphorically implies blame, so it was classified as 'no'.","The phrase 'kraesse' does not indicate a location but rather has a figurative meaning related to blame, so it is not classified as an adverbial of place."
291,rollist,pääsema,välja,el,"Paratamatult toimib siin kirikukord ja traditsioon ning see on üks näide sellest , kuidas ma ei pääse sellest rollist välja , mis mul on .",The term 'rollist' refers to a role or position and is not denoting a physical location.,"The phrase 'rollist' does not indicate a physical or locational place but rather a metaphorical or abstract concept, so it is not an adverbial of place."
425,siibrisse,hakkama,NaN,ill,"Tõsiselt siibrisse hakkab viskama see jama , kurat raisk , kui ma sinna kuju ette satun ja mingi debiilik karjub « maha okupandid » või « fasistid » siis annan valimatult pasunasse .",The phrase 'siibrisse' was classified as 'no' because it does not refer to a specific geographic or locational entity.,The phrase 'siibrisse' is not indicating a place; it is used metaphorically and does not function as an adverbial of place.
602,kraesse,hakkama,NaN,ill,""" Need , kes alates esimesest jaanuarist on käitunud sama skeemi alusel , mida Hiit kasutas , on nüüd õiged mehed ning nende kraesse ei hakka keegi .","The phrase 'kraesse' refers to a metaphorical concept or area, not a physical location.","The phrase 'kraesse' refers to something metaphorical ('on their neck') and does not indicate a physical location, so it is classified as 'no'."
645,teemasse,ütlema,NaN,ill,"ütlen siis ka siia teemasse , et nägin iksi ja kingaudiovenda .","The phrase 'teemasse' refers to a topic or subject in conversation, not a physical location, so it was classified as 'no'.",The phrase 'teemasse' is not classified as an adverbial of place because it refers to a metaphorical topic or context rather than a physical location.
...,...,...,...,...,...,...,...
8309,kerglustesse,kaduma,NaN,ill,"Tuleb loota , et selle lipu kõrgel hoidmine kitšlikult lihtsustatud maailma kerglustesse ei kao .","The phrase 'kerglustesse' refers to an abstract concept of ease or lightness, not a physical location, so it is classified as 'no'.","The phrase 'kerglustesse' refers to a figurative concept or abstract idea rather than a physical or locational place, so it is not classified as an adverbial of place."
8407,aupaistesse,lennutama,NaN,ill,"Finaalmatši kangelased on mõlemad Linnu talu tooted Linnu L ja Linnu M. Üha harvemaks , kuid teravamaks muutunud munakangelaste sööstud tipnesid finaalis Triinu osava ründe tõrjumise ja ühe tabava süütu kõksukesega , mis Linnu talu M-i pealtvaatajate võileivale , võitja aga uhkesse aupaistesse lennutab .",The phrase 'aupaistesse' was classified as not a location because it refers to 'glory' or 'fame' and not a geographical place.,The phrase 'aupaistesse' was classified as 'no' because it does not denote a physical or conceptual location or place but rather a figurative state of glory or honor.
9612,hädades,õpetama,NaN,in,"On selleks üksnes uued parlamendivalimised või teeb oma tööd lihtsalt aeg , mis kogetud hädades nii rahvast kui võimu tasapisi õpetab ?","The word 'hädades' refers to troubles or difficulties, not a specific location.","The phrase 'hädades' refers to hardships or difficulties rather than a physical or metaphorical place, so it is not adverbial of place."
9637,aujärjele,naasma,NaN,all,"Kuid palju on ju juba räägitud "" uue kesk-aja "" saabumisest praeguses ekraaniühiskonnas , kus pildilisus on naasnud sajanditetagusele aujärjele .","The phrase 'aujärjele' metaphorically refers to a position of honor or status, not a physical location, so it is classified as 'no'.","The phrase 'aujärjele' metaphorically describes a status or position, not a physical 

In [92]:
no_ex[(no_ex["explanation2"].str.contains("subject"))]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
249,bioloogias,jagama,NaN,in,"Tema võitudepagas on raske : teine koht matemaatikas , esikoht füüsikas , esikoht keemias ( vabariiklikul võistlusel seitsmes koht 25 võistleja hulgas ) , geograafias teine koht ( vabariiklikul võistlusel 1. järgu diplom 23 osaleja seast ) , ajaloos esimene ( vabariiklikul võistlusel 21. , osales 24 võistlejat ) , bioloogias jagas kolmandat kuni viiendat kohta , loodusteaduses oli vabariigis 20. , osales 34 võistlejat .",?,"The phrase 'bioloogias' indicates the subject or discipline and does not describe a location, so it is not an adverbial of place."
379,küsimustes,kauplema,NaN,in,"Muidugi kauplesime kõvasti aktsiaemissiooni küsimustes , aga sellest said minu arust mõlemad pooled aru kui äri normaalsest protseduurist .","The word 'küsimustes' refers to issues or topics, not a physical or geographical location.","The phrase 'küsimustes' refers to a topic or subject matter, not a location, so it is not classified as an adverbial of place."
1109,EMU-küsimuses,valitsema,NaN,in,Soome parlamendi liikmete hulgas valitseb EMU-küsimuses veel segadus .,"The phrase 'EMU-küsimuses' refers to a topic of discussion, not a geographic or physical location.","The phrase 'EMU-küsimuses' refers to a topic or subject matter rather than a location, so it is not classified as an adverbial of place."
1180,Arvis,ehitama,NaN,in,"Arvis polnud seda aga ise ehitanud , vaid Saksamaalt ostnud .",The phrase 'Arvis' refers to a person's name and not a location.,The phrase 'Arvis' was classified as 'no' because it refers to a subject (a person) and does not indicate a location or place.
1761,Pathfinderi,panema,NaN,adit,"Ligi 24 cm kliirens ning elektrooniline kaapeväldik lubaksid päris kaugele rappa sumbata , kuid piiri panevad lõpuks Pathfinderi ikkagi teeliikluseks mõeldud rehvid .",The word 'Pathfinderi' refers to a car model and does not specify a geographic location.,"The phrase 'Pathfinderi' refers to the subject, not a location, so it is not an adverbial of place."
1810,teemasse,käima,NaN,ill,Kas tunnetamine käib ka teemasse ?,"'teemasse' was classified as 'no' because it refers to an abstract notion or topic, not a physical location.","The phrase 'teemasse' was classified as not adverbial of place ('no') because it refers to a topic or subject, not a location."
3271,Kairis,möllama,NaN,in,"Kairis möllasid korterisse sisenedes vastakad tunded : « Vaatasin , autot maja ees pole .",NaN,The phrase 'Kairis' was classified as 'no' because it functions as the subject of the sentence and does not indicate a place.
3613,rahast,suunduma,NaN,el,"Lõviosa Eestis aastatel 1996-1999 Via Baltica tarvis kuluvast rahast suundub hoopis Tallinna , kus kasutust leiab umbes 25 miljonit dollarit .",The word 'rahast' refers to money or funding and does not denote a location.,"The phrase 'rahast' is not an adverbial of place because it refers to the subject of funding, which is not a location."
3857,leitmotiivi,kandma,NaN,adit,Kevadisel Genfi autonäitusel rahva ette toodava sõiduki disain on kaunis ja kannab praeguse viienda seeria auto leitmotiivi .,The phrase 'leitmotiivi' refers to a theme or motif and does not indicate a physical location.,"The phrase 'leitmotiivi' is not adverbial of place because it refers to the design theme or subject matter of the vehicle, not a location."
4407,tehnilises,rajama,NaN,in,Ilmselt oli põhjus tehnilises ja mägises rajas .,"The word 'tehnilises' refers to a quality or characteristic, not a specific location.",The phrase 'tehnilises' is not classified as an adverbial of place because it does not indicate a location but instead describes a characteristic of the subject.


In [90]:
abstrtact = no_ex[(no_ex["explanation2"].str.contains("abstract"))]
abstrtact

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
153,maaeluprobleemidesse,hakkama,NaN,ill,Sel aastal on suhtumine maaeluprobleemidesse heas mõttes muutuma hakanud .,"The phrase 'maaeluprobleemidesse' refers to rural issues rather than a specific location, so it is not classified as a location.","The phrase 'maaeluprobleemidesse' refers to an abstract concept and does not indicate a physical location, so it is not classified as an adverbial of place."
291,rollist,pääsema,välja,el,"Paratamatult toimib siin kirikukord ja traditsioon ning see on üks näide sellest , kuidas ma ei pääse sellest rollist välja , mis mul on .",The term 'rollist' refers to a role or position and is not denoting a physical location.,"The phrase 'rollist' does not indicate a physical or locational place but rather a metaphorical or abstract concept, so it is not an adverbial of place."
1179,filminduses,olema,ära,in,Vahepeal pausinduse aegu olid kõvasti filminduses ära ?,"The phrase 'filminduses' refers to the film industry and not a physical location, hence classified as 'no'.",The phrase 'filminduses' is not an adverbial of place because it refers to an abstract domain or field (filmmaking) rather than a specific physical place.
1267,eelarvamustes,eksisteerima,NaN,in,"Tehakse küll ühisnäitusi , kuid "" tüüpiline läti värk "" eksisteerib veel ainult eelarvamustes , kui siingi .","The term 'eelarvamustes' refers to a mental concept or prejudices, not a geographical or physical location, so it is classified as 'no'.",The phrase 'eelarvamustes' was classified as 'no' because it refers to a state or concept and not a physical or abstract place.
1765,kümnesse,toetama,NaN,ill,"“ South Park ” on niivõrd s ... animatsiooniga ( loe : tegelaste liikumisega ) , et see ei häirinud mind kui vaatajat no mitte üks põrm - kogu ülejäänud filmireeglistik toetab seda stiilsuse mõttes kümnesse .","The phrase 'kümnesse' is not indicative of a geographic location or a place, so it was classified as not location.","The phrase 'kümnesse' refers to an abstract concept, not a specific place, so it is not an adverbial of place."
2043,hinnangutes,jälgima,NaN,in,"Euroopa Liit seevastu jälgib oma hinnangutes Eestisse toodava suhkru koguseid ning kui need on viimase aasta jooksul olnud tavalisest märkimisväärselt suuremad , siis kehtestatakse Eestile trahv .","The phrase 'hinnangutes' refers to assessments or evaluations, not a physical or geographical location, so it was classified as 'no'.","The phrase 'hinnangutes' refers to evaluations or assessments, which are abstract and not indicative of a place, so it is not an adverbial of place."
2623,millesse,suunama,NaN,ill,"Ja muide , Azarjani poolt peilitud läkitus , millesse vaimud oma teadmised suunasid , oli meile salvestatav ainult seetõttu , et see oli tehniliselt töödeldud infokandjaks .","The word 'millesse' is a relative pronoun meaning 'in which', and does not identify or specify a physical or geographical location.",The phrase 'millesse' was classified as 'no' because it refers to an abstract concept rather than a physical or locational place.
2988,iseendasse,olema,vaja,ill,"« Et tselluliidiravi mõjuks , on eelkõige vaja usku iseendasse .","The phrase 'iseendasse' was classified as not a location because it refers to a state of being or introspection, not a physical location.","The phrase 'iseendasse' refers to a conceptual or abstract direction (into oneself) rather than a physical place, so it is not classified as an adverbial of place."
3165,koostöösse,leidma,NaN,ill,Tegelikult ei ole Siimanni lubatud muutused valitsuskoalitsiooni suhtumises koostöösse opositsiooniga aset leidnud .,"The phrase 'koostöösse' refers to collaboration, which is an abstract concept and not a physical location.",The phrase 'koostöösse' was classified as 'no' because it refers to the abstract concept of collaboration and not to an actual place or location.
3695,Pilti,jääma,kinni,adit,"( Rahva ) kristlus jääb aga Pilti kinni , jõu

In [93]:
person = no_ex[(no_ex["explanation2"].str.contains("person"))]
person

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
59,emale,maanduma,NaN,all,"Mõneski kultuuris tähendab öösel nähtud liblikas surma , või kui äsja sünnitanud emale maandub ööliblikas , siis sureb laps .","The phrase 'emale' refers to 'to the mother' and is related to a person, not a location, so it was classified as 'no'.","The phrase 'emale' refers to a person and not a physical location, so it is not classified as an adverbial of place."
88,Naruski,laskma,NaN,adit,Mae laskis Naruski üheksasekundilisse eduseisu ja alustas siis karmi tagaajamist .,'Naruski' is not classified as a location because it appears to refer to a person's name rather than a geographical location.,"The phrase 'Naruski' was classified as 'no' because it refers to a person and not a place, so it does not indicate an adverbial of place."
140,kellesse,pidama,NaN,ill,"Kohustuslikult sisse toodud naiskolleeg , kellesse mõlemad armuma peavad , küündimatu metroojaamade esisolge , kellele koha näitamiseks tervelt poolteist tundi aega läheb , on võõrkehad .","The word 'kellesse' indicates a direction or relationship rather than a physical location, so it is classified as 'no'.","The phrase 'kellesse' refers to a person rather than a place, so it is not classified as an adverbial of place."
196,Märtin-Parki,kandma,NaN,adit,Stardis kannab Märtin-Parki Toyota numbrit 25.,The phrase 'Märtin-Parki' refers to a name or a person associated with a specific entity (Toyota) and does not explicitly denote a location.,The phrase 'Märtin-Parki' is not adverbial of place because it is a name that identifies a person and not a location.
373,Raulis,liikuma,NaN,in,"Raulis liigub kindlalt , katuseharjal kingakontsade abil tasakaalu hoides .","The phrase 'Raulis' refers to a person or name, not a place, so it is not classified as a location.","The phrase 'Raulis' refers to a person and does not indicate a location, so it is not classified as an adverbial of place."
...,...,...,...,...,...,...,...
9120,Arm-strongi,jõudma,NaN,adit,"Teisi prostituute küsitledes jõudsid politseinikud Arm-strongi jälile , kui nood rääkisid ühest ilmselt armees teeninud mehest , kes oli kutsunud neid oma autosse , seksinud seal nendega , hakanud neid siis peksma ja püüdnud kägistada .",The phrase 'Arm-strongi' was classified as 'no' because it refers to a person's name rather than a location.,The phrase 'Arm-strongi' is not an adverbial of place because it refers to a person rather than a location or spatial context.
9318,Lammi,võtma,kaasa,adit,Mõnikord võtsid varjupaiga töötajad Lammi kaasa bussiga sõitma või viisid ta metsa jalutama .,"The word 'Lammi' appears to be a proper noun, likely a name, not necessarily denoting a location in this context, so it was classified as 'no'.","The phrase 'Lammi' was classified as 'no' because it refers to a person, and not to a location, and hence it cannot function as an adverbial of place."
9679,adminni,pidama,NaN,adit,"Erinevus staatilise IP-ga hostimisel ongi just see , et see ei muutu ning sa ei pea DNS serveri adminni iga päev/nädal tüütama « kle , mul uus IP » .","The word 'adminni' refers to an administrator, which is not a specific location.","The phrase 'adminni' refers to a person (the DNS server admin), not a physical location, so it is not an adverbial of place."
9700,Carlos,avama,NaN,in,"Skoori avas 24. minutil Carlos Tenorio , teise palli saatis poolakate väravavõrku 80. minutil Agustin Delgado .","The phrase 'Carlos' refers to a person's name, not a geographic or physical location, so it is classified as 'no'.","The word 'Carlos' refers to a person, not a location, so it is not an adverbial of place."


In [95]:
no_ex[(no_ex["explanation2"].str.contains("topic")) | (no_ex["explanation2"].str.contains("matter"))]

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
379,küsimustes,kauplema,NaN,in,"Muidugi kauplesime kõvasti aktsiaemissiooni küsimustes , aga sellest said minu arust mõlemad pooled aru kui äri normaalsest protseduurist .","The word 'küsimustes' refers to issues or topics, not a physical or geographical location.","The phrase 'küsimustes' refers to a topic or subject matter, not a location, so it is not classified as an adverbial of place."
645,teemasse,ütlema,NaN,ill,"ütlen siis ka siia teemasse , et nägin iksi ja kingaudiovenda .","The phrase 'teemasse' refers to a topic or subject in conversation, not a physical location, so it was classified as 'no'.",The phrase 'teemasse' is not classified as an adverbial of place because it refers to a metaphorical topic or context rather than a physical location.
1109,EMU-küsimuses,valitsema,NaN,in,Soome parlamendi liikmete hulgas valitseb EMU-küsimuses veel segadus .,"The phrase 'EMU-küsimuses' refers to a topic of discussion, not a geographic or physical location.","The phrase 'EMU-küsimuses' refers to a topic or subject matter rather than a location, so it is not classified as an adverbial of place."
1810,teemasse,käima,NaN,ill,Kas tunnetamine käib ka teemasse ?,"'teemasse' was classified as 'no' because it refers to an abstract notion or topic, not a physical location.","The phrase 'teemasse' was classified as not adverbial of place ('no') because it refers to a topic or subject, not a location."
3179,asjades,õpetama,NaN,in,""" Nad õpetasid meid nii elementaarsetes asjades , umbes nii , et kuidas kaamera töötab . ""","The phrase 'asjades' is not classified as location because it refers to things or matters, not a physical location.","The phrase 'asjades' refers to matters or things rather than a physical location, so it is not classified as an adverbial of place."
3232,kõnealusesse,soovima,NaN,ill,"Laeva kapten , kes soovib kõnealusesse piirkonda siseneda selleks , et makrelli oma laevale ümber laadida , teatab selle liikmesriigi kontrolliasutusele , kelle vööndis ümberlaadimine toimub , kavatsetava ümberlaadimise aja ja koha kõige enam 36 ja vähemalt 24 tundi enne ümberlaadimise algust .",The phrase 'kõnealusesse' is classified as 'no' because it does not specify a location; it is a generic term indicating a referenced area but lacks any explicit geographical or spatial designation.,"The phrase 'kõnealusesse' does not specify a particular place but rather refers to a general area or topic, so it is not classified as adverbial of place."
3274,küsimuses,nappima,NaN,in,"Vallaametnik on see , kes oskab nõu anda , kui endal mõnes küsimuses mõistust napib .","The phrase 'küsimuses' means 'in the matter' or 'regarding' and does not denote a geographic or physical location, so it is classified as 'no'.",The phrase 'küsimuses' was classified as 'no' because it refers to a particular issue or matter and does not specify a location.
3857,leitmotiivi,kandma,NaN,adit,Kevadisel Genfi autonäitusel rahva ette toodava sõiduki disain on kaunis ja kannab praeguse viienda seeria auto leitmotiivi .,The phrase 'leitmotiivi' refers to a theme or motif and does not indicate a physical location.,"The phrase 'leitmotiivi' is not adverbial of place because it refers to the design theme or subject matter of the vehicle, not a location."
4611,siseasjadesse,pidama,NaN,ill,Samas ei ole opositsioonilised konservatiivid inimõiguste kohtu otsusega rahul ja peavad seda sekkumiseks perekonna siseasjadesse .,'siseasjadesse' is not a location because it refers to internal matters or affairs rather than a geographic location.,"The phrase 'siseasjadesse' refers to internal family matters and not a physical or metaphorical place, hence it is not adverbial of place."
4876,küsimuses,hoidma,kinni,in,"See lisab mulle kindlust , et hoolimata sellest , kuidas Läti kodakondsusseadusega toimib , muudab Eesti seda ega hoia selles küsimuses Lätist kindlalt kinni .","The word 'küsimuses' refers to a topic or matt

In [97]:
other3 = no_ex[~no_ex["explanation2"].str.contains('|'.join(["activity", "spatial", "direction","source", "origin", "destination", "specific place", "particular place", "event", "state", "condition", "manner", "millest", "position", "movement","project", "schedule", "mtaphor", "figurative", "abstract", "person", "subject", "topic", "matter"]))]
other3

,form,verb,verb_compound,morph_case,sentence,explanation,explanation2
107,sisusse,hakkama,NaN,ill,"See oli aeg , kus raamatuhuviline poest kõik ilmunud uudiskirjanduse koju tassis ja alles siis sisusse hakkas süvenema .","The phrase 'sisusse' refers to content or substance of something, not a physical location.","The phrase 'sisusse' was classified as 'no' because it refers to delving into the substance or content, not a physical location or place."
132,Powercolori,saama,NaN,adit,"Esmalt panin monitori VGA auku ja kaasasolnud plaadilt driverid peale , pilt imelikult väike , virvendab kui krt ( vaatamata sellele , et refresh kasvõi 200Hz keerata ) Sai installit drivereid nii Powercolori kui ka ATI saidilt ( powered by ATI ) kuid ei ss-gi , ikka sama .","The phrase 'Powercolori' refers to a brand or company name, not a location.",The word 'Powercolori' refers to a specific brand and does not indicate location or place; hence it is not an adverbial of place.
142,teaduses,edenema,NaN,in,"Ja teha igal aastal teaduspoliitikas püstitatud ülesannete täitmisest vahekokkuvõte , kuidas edenevad asjad Eesti teaduses 21. sajandi alguses .","The word 'teaduses' refers to science as a field of study, not a geographical or physical location.","The phrase 'teaduses' refers to the domain of science rather than a physical location, so it is not classified as an adverbial of place."
157,ER-i,tahtma,NaN,adit,Peale ER-i tahab praeguse seisuga uudisteagentuuri ETA osta ka Läti uudisteagentuur LETA .,"The phrase 'ER-i' refers to an entity or organization and not a physical location, so it is classified as 'no'.","The phrase 'ER-i' refers to an entity rather than a location, so it is not classified as an adverbial of place."
194,nägemisest,suunduma,NaN,el,"Näen läbi maskiklaasi umbes meetri sügavusel kivist pinnast , mis suundub minu nägemisest sügavale põhja .","'nägemisest' refers to the act of seeing and not a location, so it was classified as 'no'.","The phrase 'nägemisest' is not adverbial of place because it refers to the scope or perspective of vision, not a specific location."
...,...,...,...,...,...,...,...
9839,Kohtumises,nappima,NaN,in,Kohtumises pikakasvulise Ukraina naiskonnaga nappis Eestil jõudu korvi all ja nii tuli leppida kaotusega 69 : 72.,'Kohtumises' is not classified as a location because it refers to an abstract event (meeting or match) and not a geographical place.,"The phrase 'Kohtumises' refers to a time or context of the meeting rather than specifying a location, so it is not an adverbial of place."
9850,kirja,pidama,NaN,adit,"Osa vanalinna territooriumil paiknevaid kultuuriasutusi peab seda kirja esimeseks võimaluse kompimiseks , et tõepoolest üürihinnad tõsta turuhinna tasemele .",The word 'kirja' refers to a letter or document and not to a geographical location.,"The phrase 'kirja' refers to 'letter' or 'writing' and does not indicate a place, so it is not adverbial of place."
9859,kaubast,tellima,NaN,el,"Hulk inimesi ( sealhulgas ka ma ise ) tellivad enamiku eluks vajalikust kaubast Internetist , see on saanud oluliseks osaks igapäevasest elust .",'kaubast' does not refer to a physical place or geographical location but rather goods or items purchased online.,"The phrase 'kaubast' refers to 'goods' or 'merchandise' and does not indicate a place, so it is not adverbial of place."
9866,keskkonnaprobleemidesse,olema,vaja,ill,""" ühiskonna poolt tervishoiupraktikas omaks võetud käsitused inimesest , tervisest ja haigusest , ühiskonnast ja maailmast toetavad seisukohta , et keskkonnaprobleemidesse ei ole vaja sekkuda , seega toetavad nad valitsevat olukorda , "" tõdeb raamat nukralt .","The word 'keskkonnaprobleemidesse' refers to environmental issues, not a place or location.","The phrase 'keskkonnaprobleemidesse' is not an adverbial of place because it indicates a context or scope, not a physical location."


### muud statistikat

In [18]:
counts2 = df1.groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,count
162,pidama,NaN,149
185,ronima,NaN,117
143,ootama,NaN,110
90,kõndima,NaN,107
57,kaduma,NaN,102
...,...,...,...
106,lendama,tagasi,6
250,turustama,NaN,5
228,tarnima,NaN,5
268,viima,edasi,5


In [19]:
aggreg = df1.groupby(["verb", "verb_compound"], dropna=False).agg(
    yes_count=("classification", lambda x: np.sum(x == "yes")/len(x)*100),
        size = ("classification", lambda x :len(x))
).sort_values('yes_count', ascending=False)
aggreg

,,yes_count,size
verb,verb_compound,,
tarnima,NaN,100.000000,5
lendama,tagasi,100.000000,6
jääma,maha,100.000000,20
kolima,tagasi,100.000000,8
pussitama,NaN,100.000000,12
...,...,...,...
võtma,välja,29.545455,44
olema,ära,27.500000,40
paistma,NaN,26.829268,41


In [20]:
aggreg[aggreg["yes_count"]>= 80]

,,yes_count,size
verb,verb_compound,,
tarnima,NaN,100.0,5
lendama,tagasi,100.0,6
jääma,maha,100.0,20
kolima,tagasi,100.0,8
pussitama,NaN,100.0,12
...,...,...,...
külmuma,NaN,80.0,25
kavandama,NaN,80.0,20
pidutsema,NaN,80.0,20


In [22]:
aggreg[aggreg["yes_count"]<= 40]

,,yes_count,size
verb,verb_compound,,
ringlema,NaN,38.888889,54
panema,NaN,38.709677,31
ütlema,NaN,38.461538,26
kaevama,NaN,37.500000,16
kandma,NaN,36.507937,63
paigutama,NaN,36.170213,47
andma,NaN,35.714286,28
võimutsema,NaN,34.693878,49
tõstma,välja,33.333333,27


### varem vaadatud näited

In [24]:
ab = df1.loc[df1["verb"]=="voolama", ["form", "sentence", "classification", "explanation", "classification2", "explanation2"]]
ab

,form,sentence,classification,explanation,classification2,explanation2
119,kestast,"Paar tundi tules soojenenud ja siis lõhkenud mürsk oleks eridemineerijate hinnangul võinud veelgi traagilisemaid tagajärgi põhjustada , kuid õnneks oli osa lõhkeainet juba enne plahvatust kestast välja voolanud .",no,"The phrase 'kestast' refers to a shell or casing and is not a location, so it was classified as 'no'.",yes,NaN
217,silmadest,lili: ja voolab välja silmadest.,no,"The phrase 'silmadest' pertains to body parts (eyes) rather than a geographic or specific place, so it is not considered a location.",yes,NaN
553,hetkest,"3 ) vedeliku hulga , mis laadimiskail voolab välja suurimal laadimiskiirusel laadimisvooliku , laadimisvarre või mõne muu vedeliku pidevaks laadimiseks kasutatava seadme ( laadimisseade ) või torujuhtme täieliku purunemise hetkest kuni juurdevoolu täieliku peatamiseni .",no,"The phrase 'hetkest' refers to a point in time, not a physical location, so it was classified as 'no'.",no,"The phrase 'hetkest' refers to a point in time, not a specific location, so it is not classified as an adverbial of place."
640,Aukudest,Aukudest voolab maitsev lihamahl välja ja liha jääb kuivem .,no,"The phrase 'Aukudest' refers to holes, which are not locations in a geographic sense.",yes,The phrase 'Aukudest' is classified as an adverbial of place because it specifies a physical location from which the liquid flows.
1512,sugutist,Veri voolab sugutist välja ning erektsioon kaob .,no,The term 'sugutist' refers to a body part and not a physical place or location.,yes,NaN
2008,avaristist,"Eilseks oli avaristist välja voolanud enam kui 11 000 tonni raskeõlisid , mis on merekaldal põhjustanud tuhandete mereelukate ja lindude hukkumise .",no,"'avaristist' refers to a source of origin, not a physical or geographical location, so it is classified as 'no'.",yes,NaN
2247,piludest,Munakollane jääb nõkku pidama ja -valge voolab piludest välja .,no,"The term 'piludest' refers to openings or slots, which are not a defined location.",yes,NaN
2267,Muusikamasinast,"Muusikamasinast voolasid välja sajandialguse saksa ja eesti lööklaulud , süüa ei pidanud ka liiga kaua ootama .",no,The phrase 'Muusikamasinast' refers to a source of music and not a location.,yes,NaN
2625,rehvist,"Autojuht tunnistas , et oli oma BMW igasse rehvi mahutanud 30 liitrit piiritust täpsemal uurimisel voolas rehvist välja koguni 33 liitrit alkoholi .",no,"The word 'rehvist' refers to a car tire, which is an object rather than a location.",yes,NaN
4195,säärest,SpeaKer: säärest voolab välja,no,"The word 'säärest' refers to a body part ('shank') and not a specific location, so it is classified as 'no'.",yes,NaN


In [28]:
ab[(ab["classification2"]=="no")]

,form,sentence,classification,explanation,classification2,explanation2
553,hetkest,"3 ) vedeliku hulga , mis laadimiskail voolab välja suurimal laadimiskiirusel laadimisvooliku , laadimisvarre või mõne muu vedeliku pidevaks laadimiseks kasutatava seadme ( laadimisseade ) või torujuhtme täieliku purunemise hetkest kuni juurdevoolu täieliku peatamiseni .",no,"The phrase 'hetkest' refers to a point in time, not a physical location, so it was classified as 'no'.",no,"The phrase 'hetkest' refers to a point in time, not a specific location, so it is not classified as an adverbial of place."


In [101]:
df = df1[["form", "lemma", "verb", "verb_compound", "morph_case", "sentence", "classification", "classification2", "explanation2"]]
idx1 = (df["classification"]=="no")

df[idx1 & (df["form"]=="näkku")]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
3831,näkku,nägu,vedama,NaN,adit,"Aastad on Henno pruuniks parkunud näkku vagusid vedanud , metsa all toimetades meenutab ta Eno Raua lasteraamatu kangelast Sammalhabet - vaid linnupesa on veel pikast hallisegusest habemest puudu .",no,yes,NaN
4294,näkku,nägu,mahtuma,ära,adit,"rebis: rõõm on nii suur kohe , ett ei mahu näkku ära",no,yes,NaN


In [102]:
df[idx1 & (df["form"].str.contains("hinges"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
464,hinges,hinge,protesteerima,NaN,in,"Teadsin väga hästi , kuidas ta kõiki tähttähelisi ettekirjutusi põlates nendesamade plaanide vastu oma hinges protesteeris .",no,yes,NaN
2147,hinges,hing,helisema,NaN,in,"Minu hinges heliseb lapsepõlves Leida Lepajõelt kuuldud jutustus sellest , kuidas nad , mõned hulljulged Pärnu gümnaasiumi tüdrukud , aiakäru rattad koidikueelsel Pärnu munakivisillutisel tärisemas , käisid päästmas-peitmas äsja õhitud Amandus Adamsoni vabadussamba säilinud detaili , Poiss lilledega ...",no,yes,NaN
2562,hinges,hing,pesitsema,NaN,in,"Coulthardi hinges pesitseb okas mulluse kaotuse pärast tiimikaaslasele , ta januneb revanshi järele .",no,yes,NaN
4462,hingest,hinge,saama,välja,el,Et saaks hingest välja .,no,yes,NaN
4907,hinges,hing,laiutama,NaN,in,"Täna tunneb Valeri Repson midagi sellist , mida ta juba ammu kogenud pole — tema hinges laiutab tiibu hoopis võimutunne .",no,yes,NaN
7511,hinges,hinge,arenema,NaN,in,eks me vaikselt vast areneme oma hinges .,no,yes,The phrase 'hinges' was classified as 'yes' because it indicates a metaphorical place (in one's soul).


In [103]:
df[idx1 & (df["form"].str.contains("koju"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
1626,koju,kodu,pagema,NaN,adit,Seekord pistsid poisid jooksu ja pagesid lähemal elava sõbra juurde koju .,no,yes,NaN
9420,koju,kodu,reisima,NaN,adit,Asendusliige Janno Simm tuleb jahiga Euroopasse ja reisib sealt koju .,no,yes,"The phrase 'koju' specifies direction towards a location (home), so it was classified as an adverbial of place ('yes')."


In [104]:
df[idx1 & (df["form"].str.contains("Liidus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
582,Liidust,Liit,saama,välja,el,"Ühe hinna järgi saavad odavat viina Euroopa Liidust välja sõitjad , teise järgi kallimat viina Euroopa Liitu sisse sõitjad .",no,yes,NaN
684,Liidus,liit,opereerima,NaN,in,"Euroopa Liidus opereerib Falck suurimat kiirabiteenistust ja on ainuke kontsern , kes pakub erakorralise meditsiini teenust oma koduriigist väljaspool .",no,yes,NaN
1139,Liidus,Liidu,liikuma,NaN,in,"Bulgaaria , Rumeenia ja Sloveenia on juba Euroopa Liidus ning NATOs , Horvaatia , Makedoonia , Bosnia- ja Herzegoviina liiguvad samas suunas .",no,yes,NaN
2203,Liidus,liit,jooma,NaN,in,"Nõukogude Eestis elas üks nn raamaturahvas , kes seda maad külastanud türgi luuletaja Nazõm Hikmeti sõnul “ luges kõige rohkem luulet ja jõi kõige rohkem viina ” terves Nõukogude Liidus .",no,yes,NaN
2649,Liidust,Liidu,saama,välja,el,siis kui Eesti sai N Liidust välja oli mitu ärevat hetke .,no,yes,NaN
3325,Liidus,Liidud,seisma,NaN,in,"Seetõttu tuleb meil praegu endale väga selgelt ja ilma igasuguse roosamannata teadvustada , et nii Euroopa Liidus kui ka NATO-s seisab meil esimese ülesandena päevakorras enda maksmapanek , oma reviiri mahamärkimine .",no,yes,NaN
8729,Liidus,Liidu,ootama,NaN,in,Maalehe ja Põllumajandus-Tööstuskoja korraldatud konverentsi Aasta põllumees 2002 raames toimunud väitlus teemal Mis meid ootab Euroopa Liidus ?,no,yes,NaN
9702,Liidus,Liidu,toimima,NaN,in,"Arvamustevahetust tervitades peame ometi tagasi lükkama ettepanekud , milles soovitatakse kasutada endises N Liidus toiminud suhtlemisskeeme ja -võtteid .",no,yes,NaN


In [105]:
df[idx1 & (df["form"].str.contains("nimetus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
3133,nimetusse,nimetus,viskama,NaN,adit,Vanim poeg Sulev viskas kivi kaugele-kaugele mingisse nimetusse järve .,no,no,The phrase 'nimetusse' was classified as 'no' because it describes a characteristic (nameless) of the object 'järve' rather than specifying a place.


In [106]:
df[idx1 & (df["form"].str.contains("pimeda"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
8609,pimedast,pime,pääsema,välja,el,"Esialgu on veel vara ennustada , kas too rändur sealt pimedast enam välja pääsebki .",no,yes,NaN


In [107]:
df[idx1 & (df["form"].str.contains("peaaegu-pimeduses"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
1587,peaaegu-pimeduses,peaaegu-pimedus,kõndima,NaN,in,"Me kõndisime soojas peaaegu-pimeduses jaama poole ja ma aimasin laiduväärsushäbi ja võidurõõmuga ette , kuhu see kõndimine viib .",no,no,"The phrase 'peaaegu-pimeduses' is not an adverbial of place because it describes the condition or lighting of the environment, not a specific location."


In [108]:
df[idx1 & (df["form"].str.contains("laag"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
7411,merelaagrisse,merelaager,ootama,NaN,ill,Pirita vaba aja keskus ootab 10.-14. juunini 7-12aastasi lapsi uudishimulike kunstnike laagrisse ( osavõtutasu 250 krooni ) ning 29. juulist 2. augustini 7-14aastasi merelaagrisse .,no,yes,NaN


In [109]:
df[idx1 & (df["form"].str.contains("Kõvaketas"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
1586,Kõvaketast,kõvaketa,röövima,NaN,el,-> Kõvaketast röövib ca 9 giga .,no,no,The phrase 'Kõvaketast' is not an adverbial of place because it refers to the object being acted upon rather than indicating a location.


In [110]:
df[idx1 & (df["form"].str.contains("rööbas"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
4138,rööbastesse,rööbas,pöörama,NaN,ill,"Pärast seda pöörab elu taas normaalsetesse rööbastesse , "" lausus Adlas .",no,yes,NaN


In [111]:
df[idx1 & (df["form"].str.contains("käekot"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
989,käekotist,käekott,varastama,NaN,el,"Kasutades kannatanu abitut seisundit , varastas kõrvaltoast käekotist 500",no,yes,NaN


In [112]:
df[idx1 & (df["form"].str.contains("kirja"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
115,lastekirjandusse,lastekirjandus,laienema,NaN,adit,See atmosfäär laienes ka lastekirjandusse .,no,yes,NaN
146,kirjast,kiri,kostma,NaN,el,"Võimalik , et te seda praegu veel pole , sest kirjast kostab rohkem ahastust sel teemal , mida teie eakaaslased on juba saavutanud .",no,yes,NaN
158,ajakirjanduses,ajakirjandus,vastutama,NaN,in,Ametlikult vastutab ajakirjanduses avaldatu eest väljaandja .,no,yes,NaN
272,ajakirjandusse,ajakirjandus,laskma,NaN,adit,"Lukas ütles , et lase asi ajakirjandusse , vaatame , mis juhtub .",no,yes,NaN
426,kirjas,kiri,levitama,NaN,in,"Kas Gustav Lükk tõesti levitas oma kirjas sõjaväevõimude kohta vale ja kuritahtlikke teateid , on tagantjärele raske öelda .",no,no,"The phrase 'kirjas' describes the medium of communication instead of a specific location, so it is not an adverbial of place."
470,sisustusajakirjas,sisustusajakiri,kohama,NaN,in,"Iseenesest ju ilus karp , ja kui nüüd hästi järele mõelda , kas polnud ta just selliseid kämpe , seitsmekümnendaid meenutavaid nipsasju hiljuti ühes välismaises sisustusajakirjas kohanud ?",no,yes,The phrase 'sisustusajakirjas' was classified as 'yes' because it describes the location (in an interior design magazine) where something was encountered.
475,ajakirjandusest,ajakirjandus,jõudma,tagasi,el,"Esimene vaatab , et info jõuaks valitsusest ajakirjandusse , teine omakorda , et info ajakirjandusest valitsusse tagasi jõuaks .",no,yes,NaN
941,ajakirjanduses,ajakirjandus,jagama,NaN,in,"Patric Marberi Broadway hitist "" Closer "" ( tööpealkirjaga "" Kes keda ? "" ) on ajakirjanduses muljeid jaganud Merle Karusoo lavastusega "" Kured läinud ... "" Ameerikas käinud seltskond .",no,yes,NaN
957,ajakirjandusse,ajakirjandus,saatma,NaN,adit,"Venemaa Tshetsheenia väegrupeeringu juhatus saatis eile ajakirjandusse sõnumi , et Gruusia piiri tagant valmistub iseseisvuslastele appi tulema suur hulk professionaalselt relvastatud Afganistani valitseva islamiliikumise Talebani võitlejaid .",no,yes,NaN
1214,ajalookirjanduses,ajalookirjandus,laiuma,NaN,in,"Ühtlasi tuleb muidugi tõdeda , et käesolev teos , ükskõik kui hinnatav see oma nišis ka poleks , ei suuda ometi täita kogu seda kuristikku , mis meie ajalookirjanduses kas või Venemaa suunalgi laiub .",no,yes,NaN


In [113]:
df[idx1 & (df["form"].str.contains("pagendus"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
9000,pagendusse,pagendus,pidama,NaN,adit,"Millegipärast enamik represseeritute ühinguid ei pea neid pagendusse läinuteks , vaid peab repressioonide eest põgenejateks .",no,yes,"The phrase 'pagendusse' is classified as an adverbial of place because it indicates movement towards a specific location, namely exile."


In [114]:
df[idx1 & (df["form"].str.contains("liin"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
1360,elektriliinist,elektriliin,varastama,NaN,el,"Elektrit varastasid "" vabrikandid "" naabertalu elektriliinist .",no,yes,The phrase 'elektriliinist' is classified as an adverbial of place because it specifies the physical location (electric line) where the action occurred.
2955,liinilt,liin,väljuma,NaN,abl,Land Cruiseri legendi 50. aastapäeva tähistamiseks väljusid Toyotal liinilt nüüd kaks Land Cruiseri juubeliaasta erimudelit .,no,yes,NaN
3222,Eluliini,eluliin,pöörduma,NaN,adit,"2000. aastal pöördus Eluliini 4003 inimest , kellest 1000 olid valmis elust loobuma .",no,yes,NaN
3666,eesliinile,eesliin,ronima,NaN,all,"“ Ta on tark , temaga on huvitav rääkida , ” kiidab inspektor ja lisab samas , et limonovlaste liider on kaval kuju , kes koordineerib tegevust ja juhatab oma jüngreid , kuid ise eesliinile ei roni .",no,yes,NaN
6813,elektriliinidesse,elektriliin,looma,NaN,ill,"Eile varahommikul lõi äike elektriliinidesse Ida-Virumaal Püssi alajaama juures , mis jättis Eesti Energia teatel ligi tunniks ajaks vooluta AS Flexa puidutööstuse Viru-Nigulas ja paljud väiketarbijad .",no,yes,NaN
7915,liinile,liin,pöörduma,tagasi,all,"Me lihtsalt pöördume tagasi sellelesamale lõputute vaidluste liinile ning tulemus on see , et me ei ehita valmis ühtegi objekti .",no,yes,NaN


In [115]:
df[idx1 & (df["form"].str.contains("That"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
7308,That's,That,lindistama,NaN,in,"Elvis , Scotty Moore ja Bill Black lindistasid "" That's all right "" -nimelise laulu Sun Recordsile 5. juulil 1954. aastal .",no,no,The phrase 'That's' is not an adverbial of place because it is part of a title and does not refer to a location.


In [116]:
df[idx1 & (df["form"].str.contains("sfäär"))]

,form,lemma,verb,verb_compound,morph_case,sentence,classification,classification2,explanation2
106,sfääris,sfäär,ringlema,NaN,in,"Kuna olen naftat puurinud ja geoloogilistel ekspeditsioonidel käinud , kõik mu noorepõlve aastad on ju rännakute rõõmud , siis olen selles sfääris ringelnud .",no,yes,NaN
2053,atmosfääris,atmosfäär,laiutama,NaN,in,Vene suurriiklus laiutab atmosfääris nagu eeter .,no,yes,NaN
4161,erasfääri,erasfäär,liikuma,NaN,adit,"Vabanemisel liiguksid nad erasfääri , kus teatud valdkondades , alates teatud tasemest , valitseb tugev töökäte puudus .",no,yes,"The phrase 'erasfääri' specifies a location where a certain condition prevails, qualifying it as an adverbial of place."
5730,mõjusfääri,mõjusfäär,liikuma,NaN,adit,Politseisiseste motivatsiooni- ja distsipliiniprobleemide tõttu kaotab riik kontrolli politsei üle ja see liigub organiseeritud kuritegelike struktuuride mõjusfääri .,no,yes,"The phrase 'mõjusfääri' indicates a specific place or location where something is being influenced or moved towards, making it an adverbial of place."
6452,ametisfääris,ametisfäär,vastutama,NaN,in,"Kas te jagate minu arusaamist , et ministrid ei vastuta ikkagi ( mitte nii , nagu Kalle Jürgenson lapsemeelselt väitis ) isiklike tegude eest , vaid vastutavad oma ametisfääris toimuva eest ?",no,yes,NaN
